<div style=" background-color: RGB(0,114,200);" >
<h1 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">PROJET 9 DATA ANALYST</h1>
<h2 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">Analysez les ventes d'une librairie avec R ou Python</h2>
</div>

<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1. Téléversement des fichiers</h2>
</div>

In [1]:
#importer lib
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import shapiro

In [2]:
#chargement données
costumers = pd.read_csv(r'customers.csv',sep=";")
products = pd.read_csv(r'products.csv',sep=";")
transactions = pd.read_csv(r'Transactions.csv',sep=";")

In [3]:
print('Trasction shape:', transactions.shape)
transactions.head(5)

Trasction shape: (687534, 4)


,id_prod,date,session_id,client_id
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033


In [4]:
transactions.describe(include='all')

,id_prod,date,session_id,client_id
count,687534,687534,687534,687534
unique,3265,687419,345505,8600
top,1_369,2021-10-06 21:29:59.752585,s_118668,c_1609
freq,2340,2,14,25586


In [5]:
print("costumers shape:", costumers.shape)
print('numeri di clienti unici:', costumers['client_id'].nunique())
costumers.head(5)

costumers shape: (8621, 3)
numeri di clienti unici: 8621


,client_id,sex,birth
0,c_4410,f,1967
1,c_7839,f,1975
2,c_1699,f,1984
3,c_5961,f,1962
4,c_5320,m,1943


In [6]:
#age dans costumers
costumers['age'] = 2023 - costumers['birth']
#ajouter tranches d'ages : 18-24studenti, primo lavoro, 25-34giovani professionisti 35-49adulti con famiglia, 50-64 pre-pensionamento, potere d'acquisto alto, 65+pensionati, lettori forti
# 18-24 / 25-34 / 35-49 / 50-64 / 65+

conditions = [
    costumers['age'] < 25,
    costumers['age'] < 35,
    costumers['age'] < 50,
    costumers['age'] < 65,
]

labels = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités']

costumers['tranche_age'] = np.select(conditions, labels, default='65+ Retraités')
costumers.head()

,client_id,sex,birth,age,tranche_age
0,c_4410,f,1967,56,50-64 Pré-retraités
1,c_7839,f,1975,48,35-49 Adultes
2,c_1699,f,1984,39,35-49 Adultes
3,c_5961,f,1962,61,50-64 Pré-retraités
4,c_5320,m,1943,80,65+ Retraités


In [7]:
costumers.describe(include='all')

,client_id,sex,birth,age,tranche_age
count,8621,8621,8621.000000,8621.000000,8621
unique,8621,2,NaN,NaN,5
top,c_4410,f,NaN,NaN,35-49 Adultes
freq,1,4490,NaN,NaN,2633
mean,NaN,NaN,1978.275606,44.724394,NaN
std,NaN,NaN,16.917958,16.917958,NaN
min,NaN,NaN,1929.000000,19.000000,NaN
25%,NaN,NaN,1966.000000,31.000000,NaN
50%,NaN,NaN,1979.000000,44.000000,NaN
75%,NaN,NaN,1992.000000,57.000000,NaN


In [8]:
#puizia del client_id
costumers['client_id'] = costumers['client_id'].str.replace('c_', '').astype(int)
transactions['client_id'] = transactions['client_id'].str.replace('c_', '').astype(int)

In [9]:
#vérificiation avant merge
transactions.shape, costumers.shape

((687534, 4), (8621, 5))

In [10]:

#merged tra transactions e costumers per avere age e tranche_age nel df transactions
transactions = transactions.merge(costumers[['client_id', 'age', 'tranche_age']], on= 'client_id', how='left' )
print(transactions.shape)
print(transactions.isnull().sum())
transactions.head()



(687534, 6)
id_prod        0
date           0
session_id     0
client_id      0
age            0
tranche_age    0
dtype: int64


,id_prod,date,session_id,client_id,age,tranche_age
0,0_1259,2021-03-01 00:01:07.843138,s_1,329,56,50-64 Pré-retraités
1,0_1390,2021-03-01 00:02:26.047414,s_2,664,63,50-64 Pré-retraités
2,0_1352,2021-03-01 00:02:38.311413,s_3,580,35,35-49 Adultes
3,0_1458,2021-03-01 00:04:54.559692,s_4,7912,34,25-34 Jeunes professionnels
4,0_1358,2021-03-01 00:05:18.801198,s_5,2033,67,65+ Retraités


In [11]:
print('numeri di clienti unici:', costumers['client_id'].nunique())


numeri di clienti unici: 8621


In [12]:
#répartition du chiffre d'affaires pour les clients BtoB

achats_par_client = (transactions.groupby('client_id')['id_prod']
                     .count()
                     .reset_index(name='nb_achats')
                     .sort_values('nb_achats', ascending=False))

print(achats_par_client['nb_achats'].describe())
achats_par_client.head(5)

count     8600.000000
mean        79.945814
std        313.532311
min          1.000000
25%         28.000000
50%         52.000000
75%         97.000000
max      25586.000000
Name: nb_achats, dtype: float64


,client_id,nb_achats
1604,1609,25586
6698,6714,9199
3446,3454,6793
4944,4958,5222
2135,2140,405


In [13]:
# Quanti clienti superano diverse soglie?
for soglia in [300, 400, 500, 750, 1000]:
    n = len(achats_par_client[achats_par_client['nb_achats'] > soglia])
    print(f'Clients avec plus de {soglia} achats : {n}')

print('----------------------')

#seuil b2b majeur 500 acahts sur 2ans
clients_b2b = achats_par_client[achats_par_client['nb_achats'] > 500]
print(clients_b2b)




Clients avec plus de 300 achats : 130
Clients avec plus de 400 achats : 6
Clients avec plus de 500 achats : 4
Clients avec plus de 750 achats : 4
Clients avec plus de 1000 achats : 4
----------------------
      client_id  nb_achats
1604       1609      25586
6698       6714       9199
3446       3454       6793
4944       4958       5222


In [14]:
#df costumers b2b - 4 clienti business 1609, 3454, 4958, 6714
df_costumers_b2b = costumers[costumers['client_id'].isin([1609, 6714, 3454, 4958])]

#df costumers b2c no costumers
df_costumers_b2c = costumers[~costumers['client_id'].isin([1609, 6714, 3454, 4958])]

#liste b2b
liste_b2b = df_costumers_b2b['client_id'].tolist()
print(liste_b2b)


[4958, 6714, 1609, 3454]


In [15]:
df_costumers_b2b.head(10)

,client_id,sex,birth,age,tranche_age
1378,4958,m,1999,24,18-24 Étudiants
1911,6714,f,1968,55,50-64 Pré-retraités
3641,1609,m,1980,43,35-49 Adultes
8087,3454,m,1969,54,50-64 Pré-retraités


In [16]:
print("products shape:", products.shape)
products.head(5)

products shape: (3286, 3)


,id_prod,price,categ
0,0_1421,19.99,0
1,0_1368,5.13,0
2,0_731,17.99,0
3,1_587,4.99,1
4,0_1507,3.99,0


In [17]:
products.describe(include='all') 

,id_prod,price,categ
count,3286,3286.000000,3286.000000
unique,3286,NaN,NaN
top,0_1421,NaN,NaN
freq,1,NaN,NaN
mean,NaN,21.863597,0.370359
std,NaN,29.849786,0.615446
min,NaN,0.620000,0.000000
25%,NaN,6.990000,0.000000
50%,NaN,13.075000,0.000000
75%,NaN,22.990000,1.000000


In [18]:

transactions['session_id'] = transactions['session_id'].str.replace('s_', '').astype(int)
transactions.head(5)

,id_prod,date,session_id,client_id,age,tranche_age
0,0_1259,2021-03-01 00:01:07.843138,1,329,56,50-64 Pré-retraités
1,0_1390,2021-03-01 00:02:26.047414,2,664,63,50-64 Pré-retraités
2,0_1352,2021-03-01 00:02:38.311413,3,580,35,35-49 Adultes
3,0_1458,2021-03-01 00:04:54.559692,4,7912,34,25-34 Jeunes professionnels
4,0_1358,2021-03-01 00:05:18.801198,5,2033,67,65+ Retraités


In [19]:
#ordonner les transactions par session_id
transactions = transactions.sort_values(by='session_id')
transactions.head(10)

,id_prod,date,session_id,client_id,age,tranche_age
0,0_1259,2021-03-01 00:01:07.843138,1,329,56,50-64 Pré-retraités
1,0_1390,2021-03-01 00:02:26.047414,2,664,63,50-64 Pré-retraités
2,0_1352,2021-03-01 00:02:38.311413,3,580,35,35-49 Adultes
29,0_1110,2021-03-01 00:38:57.630675,3,580,35,35-49 Adultes
11,0_1638,2021-03-01 00:10:37.223732,3,580,35,35-49 Adultes
3,0_1458,2021-03-01 00:04:54.559692,4,7912,34,25-34 Jeunes professionnels
17,1_310,2021-03-01 00:17:11.089942,4,7912,34,25-34 Jeunes professionnels
4,0_1358,2021-03-01 00:05:18.801198,5,2033,67,65+ Retraités
15,0_1475,2021-03-01 00:16:16.649539,6,4908,42,35-49 Adultes
7,0_279,2021-03-01 00:07:48.507530,6,4908,42,35-49 Adultes


In [20]:
transactions['date'] = pd.to_datetime(transactions['date']).dt.normalize()
transactions.head(5)

,id_prod,date,session_id,client_id,age,tranche_age
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités
2,0_1352,2021-03-01,3,580,35,35-49 Adultes
29,0_1110,2021-03-01,3,580,35,35-49 Adultes
11,0_1638,2021-03-01,3,580,35,35-49 Adultes


In [21]:
#vérification avant merge n° righe = df1 (celui left)
transactions.shape, products.shape 

((687534, 6), (3286, 3))

In [22]:
#ajout 'price au id_prod qui provient du df products (merged)
transactions = transactions.merge(products[['id_prod', 'price']], on='id_prod', how='left')
transactions.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46


In [23]:
#verification des valeurs apres merged
print('Transaction shape:', transactions.shape)
print(transactions.isnull().sum())
print('lines avant merged', len(transactions))



Transaction shape: (687534, 7)
id_prod        0
date           0
session_id     0
client_id      0
age            0
tranche_age    0
price          0
dtype: int64
lines avant merged 687534


In [24]:
# Converti in datetime
transactions['date'] = pd.to_datetime(transactions['date']).dt.normalize()

# Anno e mese — sistema solare
transactions['year'] = transactions['date'].dt.year
transactions['month_num'] = transactions['date'].dt.month
transactions['month'] = transactions['date'].dt.month_name()
transactions['day'] = transactions['date'].dt.day

# Settimana — sistema ISO ma con chiave combinata anno-settimana
iso = transactions['date'].dt.isocalendar()
transactions['week'] = iso.week
transactions['year_week'] = (iso.year.astype(str) + '-S' + 
                              iso.week.astype(str).str.zfill(2))

In [25]:
#top par id prod
transactions['category_id'] = transactions['id_prod'].str.split('_').str[0]
transactions['product_num'] = transactions['id_prod'].str.split('_').str[1]
transactions.head(5)


,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638


In [26]:
transactions["product_num"] = transactions["product_num"].astype(int)
transactions["category_id"] = transactions["category_id"].astype(int)

In [27]:
transactions.info()
print(transactions['price'].max())
print(transactions['price'].min())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687534 entries, 0 to 687533
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id_prod      687534 non-null  object        
 1   date         687534 non-null  datetime64[ns]
 2   session_id   687534 non-null  int64         
 3   client_id    687534 non-null  int64         
 4   age          687534 non-null  int64         
 5   tranche_age  687534 non-null  object        
 6   price        687534 non-null  float64       
 7   year         687534 non-null  int32         
 8   month_num    687534 non-null  int32         
 9   month        687534 non-null  object        
 10  day          687534 non-null  int32         
 11  week         687534 non-null  UInt32        
 12  year_week    687534 non-null  object        
 13  category_id  687534 non-null  int64         
 14  product_num  687534 non-null  int64         
dtypes: UInt32(1), datetime64[ns](1), f

In [28]:
df_panier = (transactions.groupby('session_id')
             .agg({'price': ['sum', 'count'], 'client_id': 'first', 'age': 'first', 'tranche_age':'first', 'date': 'first',
                    'day': 'first', 'week': 'first', 'month': 'first',
                    'year': 'first', 'month_num': 'first', 'year_week': 'first'}).reset_index()
                    )
df_panier.columns = ['session_id', 'total_price', 'total_items',
                      'client_id', 'age', 'tranche_age', 'date', 'day', 'week', 'month', 
                      'year', 'month_num', 'year_week'
                     ]
df_panier.head(5)

#formato data annee-mois-jour

,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
0,1,11.99,1,329,56,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
1,2,19.37,1,664,63,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
2,3,14.67,3,580,35,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
3,4,20.75,2,7912,34,25-34 Jeunes professionnels,2021-03-01,1,9,March,2021,3,2021-S09
4,5,16.49,1,2033,67,65+ Retraités,2021-03-01,1,9,March,2021,3,2021-S09


In [29]:
#test unicità client_id
print('Nombre de clients qui ont effectué des transactions :', transactions['client_id'].nunique())
print('Total de clients dans la liste :', len(costumers["client_id"]))
print((f'Total de clients qui n\'ont pas effectué de transactions : {(len(costumers["client_id"])) - (transactions["client_id"].nunique())}'))

Nombre de clients qui ont effectué des transactions : 8600
Total de clients dans la liste : 8621
Total de clients qui n'ont pas effectué de transactions : 21


In [30]:
# df avec seulement les clients B2B
df_transactions_b2b = transactions[transactions['client_id'].isin([1609, 6714, 3454, 4958])]
#df avec seulement les clients B2C
df_transactions_b2c = transactions[~transactions['client_id'].isin([1609, 6714, 3454, 4958])]

In [31]:
#transactions b2b
transactions_b2b = transactions[transactions['client_id'].isin(liste_b2b)]
#panier_b2b
df_panier_b2b = df_panier[df_panier['client_id'].isin(liste_b2b)]

<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">2. Analyses de données</h2>
</div>

<div style="background-color: RGB(0,150,250);" >
<h3 style="margin: auto; padding: 20px; color:#fff; ">1. Analyse du CA</h3>
</div>

<div style="background-color: RGB(0,128,128);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.1 CA - Total et moyenne mobile</h4>
</div>

In [32]:
#CA total des ventes
ca_total = df_panier['total_price'].sum()
print(f"Le chiffre d'affaires total des ventes est de : {ca_total:.2f} euros.")

#Ca b2b
ca_b2b = df_transactions_b2b['price'].sum()
print(f"Le chiffre d'affaires total des ventes B2B est de : {ca_b2b:.2f} euros. la difference de % {((ca_b2b/ca_total)*100):.2f}%")

#Ca b2c
ca_b2c = df_transactions_b2c['price'].sum()
print(f"Le chiffre d'affaires total des ventes B2C est de : {ca_b2c:.2f} euros. la difference de % {((ca_b2c/ca_total)*100):.2f}%")



Le chiffre d'affaires total des ventes est de : 12027663.10 euros.
Le chiffre d'affaires total des ventes B2B est de : 884296.09 euros. la difference de % 7.35%
Le chiffre d'affaires total des ventes B2C est de : 11143367.01 euros. la difference de % 92.65%


In [33]:
## df groupé par mois
ca_mensuel = df_panier.groupby(['year', 'month', "month_num"])['total_price'].sum().reset_index()
ca_mensuel.columns = ['year', 'month', 'month_num', 'total_ca']
ca_mensuel['total_ca'] = ca_mensuel['total_ca'].round(2)
ca_mensuel['date'] = pd.to_datetime(ca_mensuel['year'].astype(str) + '-' + 
                                     ca_mensuel['month_num'].astype(str) + '-01')
ca_mensuel = ca_mensuel.sort_values(by=['year', 'month_num'], ascending=True).reset_index(drop=True)
ca_mensuel.head(5)
print(f"la moyenne de ca mensuel est de : {ca_mensuel['total_ca'].mean():.2f} euros.")

la moyenne de ca mensuel est de : 501152.63 euros.


In [34]:
#ca b2b mensuel
ca_mensuel_b2b = df_transactions_b2b.groupby(['year', 'month', "month_num"])['price'].sum().reset_index()
ca_mensuel_b2b.columns = ['year', 'month', 'month_num', 'total_ca_b2b']
ca_mensuel_b2b['date'] = pd.to_datetime(ca_mensuel_b2b['year'].astype(str) + '-' + 
                                       ca_mensuel_b2b['month_num'].astype(str) + '-01')
ca_mensuel_b2b = ca_mensuel_b2b.sort_values(by=['year', 'month_num'], ascending=True).reset_index(drop=True)

In [35]:
ca_mensuel_b2b.head(5)
print(f"la moyenne de ca b2b mensuel est de : {ca_mensuel_b2b['total_ca_b2b'].mean():.2f} euros.")

la moyenne de ca b2b mensuel est de : 36845.67 euros.


In [36]:
moyenne = ca_mensuel['total_ca'].mean()

fig_ca = px.line(
    ca_mensuel, 
    x='date', 
    y='total_ca', 
    title='ca par mois',
    markers=True 
)

fig_ca.add_hline(
    y=moyenne,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Moyenne: {round(moyenne, 1)}',
    annotation_position='top right'
)

fig_ca.add_hline(
    y=ca_mensuel_b2b['total_ca_b2b'].mean(),
    line_dash='dash',
    line_color='blue',
    annotation_text=f'Moyenne B2B: {round(ca_mensuel_b2b["total_ca_b2b"].mean(), 1)}',
    annotation_position='top left'
)

fig_ca.show()

In [37]:
# df ca_week
ca_week = df_panier.groupby(['year_week', 'year'])['total_price'].sum().reset_index()
ca_week.columns = ['year_week', 'year', 'total_price']
ca_week['total_price'] = ca_week['total_price'].round(2)
ca_week = ca_week.rename(columns={'total_price' : 'total_ca'})
ca_week = ca_week.sort_values(by=['year', 'year_week'], ascending=True).reset_index(drop=True)

ca_week.head()

,year_week,year,total_ca
0,2021-S09,2021,110502.71
1,2021-S10,2021,105576.92
2,2021-S11,2021,112232.57
3,2021-S12,2021,109423.31
4,2021-S13,2021,107876.91


In [38]:
ca_week_b2b = df_transactions_b2b.groupby(['year_week', 'year'])['price'].sum().reset_index()
ca_week_b2b.columns = ['year_week', 'year', 'total_price']
ca_week_b2b = ca_week_b2b.sort_values(by=['year', 'year_week'], ascending=True)
ca_week_b2b.head(5)

,year_week,year,total_price
0,2021-S09,2021,8219.27
1,2021-S10,2021,7863.32
2,2021-S11,2021,8005.34
3,2021-S12,2021,8336.64
4,2021-S13,2021,8897.82


In [39]:
ca_week_b2c = df_transactions_b2c.groupby(['year_week', 'year'])['price'].sum().reset_index()
ca_week_b2c.columns = ['year_week', 'year', 'total_price']
ca_week_b2c = ca_week_b2c.sort_values(by=['year', 'year_week'], ascending=True)
ca_week_b2c.head(5)

,year_week,year,total_price
0,2021-S09,2021,102244.01
1,2021-S10,2021,97829.92
2,2021-S11,2021,104053.74
3,2021-S12,2021,101104.43
4,2021-S13,2021,98997.84


In [40]:
# df ca_day
ca_day = df_panier.groupby('date')['total_price'].sum().reset_index()
ca_day.columns = ['date', 'total_price']
ca_day['total_price'] = ca_day['total_price'].round(2)
ca_day = ca_day.sort_values(by='date', ascending=True)  
ca_day = ca_day.rename(columns={'total_price' : 'total_ca'})
ca_day.tail(5)

,date,total_ca
725,2023-02-24,15207.89
726,2023-02-25,15761.25
727,2023-02-26,16304.72
728,2023-02-27,19170.81
729,2023-02-28,18105.15


In [41]:
#df day b2b
ca_day_b2b = df_transactions_b2b.groupby('date')['price'].sum().reset_index()
ca_day_b2b.columns = ['date', 'total_price']
ca_day_b2b = ca_day_b2b.sort_values(by='date', ascending=True)
ca_day_b2b.head(5)

,date,total_price
0,2021-03-01,1096.57
1,2021-03-02,927.98
2,2021-03-03,1298.70
3,2021-03-04,1628.45
4,2021-03-05,1255.28


In [42]:
#ca_day b2c
ca_day_b2c = df_transactions_b2c.groupby('date')['price'].sum().reset_index()
ca_day_b2c.columns = ['date', 'total_price']
ca_day_b2c = ca_day_b2c.sort_values(by='date', ascending=True)
ca_day_b2c.head(5)

,date,total_price
0,2021-03-01,15468.65
1,2021-03-02,14558.47
2,2021-03-03,13899.99
3,2021-03-04,13567.62
4,2021-03-05,16216.09


In [43]:
ca_day['moyenne_mobile_7j'] = ca_day['total_ca'].rolling(window=7).mean()

fig_moy = px.line(ca_day, x='date', y= 'moyenne_mobile_7j',
              title='CA journalier - moyenne mobile sur 7 jours',
              labels={'moyenne_mobile_7j': 'CA'}

              )
fig_moy.show()

<div style="background-color: RGB(0,128,128);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.a PROJECTION DU CA à VENIR</h4>
</div>

In [44]:
from scipy.stats import linregress
import numpy as np

# Converti date in numeri per la regressione
ca_mensuel['date_num'] = np.arange(len(ca_mensuel))

slope, intercept, r_value, p_value, std_err = linregress(
    ca_mensuel['date_num'], 
    ca_mensuel['total_ca']
)

print(f'R² = {round(r_value**2, 4)}')
print(f'Tendance: {round(slope, 2)}€/mois')

# Previsione 6 mesi
future_num = np.arange(len(ca_mensuel), len(ca_mensuel)+12)
future_dates = pd.date_range(start='2023-03-01', periods=12, freq='MS')

df_forecast = pd.DataFrame({
    'date': future_dates,
    'total_ca': intercept + slope * future_num,
    'type': 'Prévision'
})

ca_mensuel['type'] = 'Réel'
df_plot = pd.concat([ca_mensuel[['date', 'total_ca', 'type']], df_forecast])

fig = px.line(df_plot, x='date', y='total_ca', color='type',
              title='CA mensuel + prévision 6 mois (régression linéaire)',
              labels={'total_ca': 'CA (€)', 'date': 'Date'})
fig.show()

R² = 0.0389
Tendance: 514.66€/mois


<div style="background-color: RGB(0,128,128);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.2 CA et CAT</h4>
</div>

In [45]:
#chiffre d’affaires par produit
top_produits = transactions.groupby('id_prod').agg(
    nb_ventes=('price', 'count'),
    ca=('price', 'sum')
).reset_index().sort_values('ca', ascending=False)

top_produits.head(10)


,id_prod,nb_ventes,ca
3096,2_159,650,94893.50
3070,2_135,1005,69334.95
3045,2_112,968,65407.76
3034,2_102,1027,60736.78
3152,2_209,814,56971.86
2619,1_395,1953,56617.47
2591,1_369,2340,56136.60
3043,2_110,865,53846.25
2607,1_383,1857,53834.43
2641,1_414,2246,53522.18


In [46]:
ca_mois_prod = transactions.groupby(['year','month', 'id_prod']).agg(
    nb_ventes=('price', 'count'),
    ca=('price', 'sum')
).reset_index().sort_values('nb_ventes', ascending=False)
ca_mois_prod.head(5)

,year,month,id_prod,nb_ventes,ca
19047,2021,November,1_412,131,2181.15
36394,2022,January,1_369,129,3094.71
36501,2022,January,1_498,125,2921.25
6911,2021,December,1_417,125,2623.75
36432,2022,January,1_417,125,2623.75


In [47]:
top_categories = transactions.groupby('category_id').agg(
    nb_ventes=('price', 'count'),
    ca=('price', 'sum')
).reset_index().sort_values('nb_ventes', ascending=False)
top_categories.head(10)

,category_id,nb_ventes,ca
0,0,415459,4419730.97
1,1,235592,4827657.11
2,2,36483,2780275.02


In [48]:
top_categories_b2b = df_transactions_b2b.groupby('category_id').agg(
    nb_ventes=('price', 'count'),
    total_ca=('price', 'sum')
).reset_index()
top_categories_b2b.head(5)

,category_id,nb_ventes,total_ca
0,0,28178,300530.28
1,1,14987,307555.25
2,2,3635,276210.56


In [49]:
#verification avant merge
top_categories_b2b.shape, top_categories.shape
print(max(len(top_categories_b2b), len(top_categories)))

3


In [50]:
# deviation entre b2b et b2c pour numero de ventes
deviation_categorie = top_categories_b2b.merge(top_categories, on='category_id', how='outer', suffixes=('_b2b', '_total'))
deviation_categorie['deviation_nb_ventes'] = deviation_categorie['nb_ventes_b2b'] / deviation_categorie['nb_ventes_total']*100
deviation_categorie.head(5)

,category_id,nb_ventes_b2b,total_ca,nb_ventes_total,ca,deviation_nb_ventes
0,0,28178,300530.28,415459,4419730.97,6.782378
1,1,14987,307555.25,235592,4827657.11,6.361421
2,2,3635,276210.56,36483,2780275.02,9.963545


In [51]:
#verification apres merge - outer  = righe >= max(len(df1), len(df2))
deviation_categorie.shape
deviation_categorie.isnull().sum()

category_id            0
nb_ventes_b2b          0
total_ca               0
nb_ventes_total        0
ca                     0
deviation_nb_ventes    0
dtype: int64

In [52]:
#somme des ventes tout regroupé
sales_sum = top_categories['nb_ventes'].sum()
print(f"Le nombre total de ventes est de : {sales_sum} ventes.")

Le nombre total de ventes est de : 687534 ventes.


In [53]:
#ca et vente par categories - 1.3
ca_mois_cat = transactions.groupby(['year','month', 'category_id']).agg(
    nb_ventes=('price', 'count'),
    ca=('price', 'sum')
).reset_index().sort_values('nb_ventes', ascending=False)
ca_mois_cat.head(5)

,year,month,category_id,nb_ventes,ca
27,2021,September,0,23142,246353.91
0,2021,April,0,19342,205222.46
6,2021,December,0,19302,206036.24
24,2021,October,0,18762,199250.83
18,2021,May,0,18501,196186.72


In [54]:

# CA per giorno e categoria
ca_jour_cat = (transactions.groupby(['date', 'category_id'])['price']
               .sum()
               .reset_index(name='ca'))

# Pivot table
ca_pivot = ca_jour_cat.pivot_table(
    index='date',
    columns='category_id',
    values='ca',
    aggfunc='sum'
).reset_index()

ca_pivot.columns = ['date', 'cat_0', 'cat_1', 'cat_2']

# Moyenne mobile 7 jours
ca_pivot['mm_cat_0'] = ca_pivot['cat_0'].rolling(window=7).mean()
ca_pivot['mm_cat_1'] = ca_pivot['cat_1'].rolling(window=7).mean()
ca_pivot['mm_cat_2'] = ca_pivot['cat_2'].rolling(window=7).mean()

# Creazione del grafico con Plotly Express
fig = px.line(
    ca_pivot,
    x='date',
    y=['mm_cat_0', 'mm_cat_1', 'mm_cat_2'],
    labels={
        'value': 'CA (€)',
        'date': 'Date',
        'variable': 'Catégorie'
    },
    title='Evolution du CA par catégorie — moyenne mobile 7 jours'
)

# Personalizzazione
fig.update_layout(
    width=1200,
    height=600,
    legend_title_text='Catégories',
    xaxis_title='Date',
    yaxis_title='CA (€)',
    hovermode='x unified'
)

# Rinominare le legende
fig.for_each_trace(lambda t: t.update(name=t.name.replace('mm_', 'Catégorie ')))

# Aggiungere griglia
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_xaxes(showgrid=False)

# Mostrare il grafico
fig.show()

<div style="background-color: RGB(0,128,128);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">analyses concernant la courbe graphique J par J </h4>
</div>

In [55]:
print('Giorno con CA massimo:', ca_day['date'].iloc[ca_day['total_ca'].idxmax()])
print('Giorno con CA minimo:', ca_day['date'].iloc[ca_day['total_ca'].idxmin()])
print('CA massimo:', ca_day['total_ca'].max())
print('CA minimo:', ca_day['total_ca'].min())

Giorno con CA massimo: 2021-10-02 00:00:00
Giorno con CA minimo: 2021-10-25 00:00:00
CA massimo: 22337.18
CA minimo: 11099.31


analyse octobre pq decente du ca ?


In [56]:
# Filtra ottobre 2021
oct_2021 = transactions[(transactions['year'] == 2021) & 
                         (transactions['month_num'] == 10)]

# Categorie vendute per giorno
cat_jour = (oct_2021.groupby(['date', 'category_id'])['price']
            .count()
            .reset_index(name='nb_ventes'))

# Totale vendite per giorno
total_jour = (oct_2021.groupby('date')['price']
              .count()
              .reset_index(name='total_ventes'))

# Merge e calcola percentuale
cat_jour = cat_jour.merge(total_jour, on='date')
cat_jour['pct'] = (cat_jour['nb_ventes'] / cat_jour['total_ventes'] * 100).round(2)

print(cat_jour.head(10))

        date  category_id  nb_ventes  total_ventes    pct
0 2021-10-01            0        663          1142  58.06
1 2021-10-01            1        441          1142  38.62
2 2021-10-01            2         38          1142   3.33
3 2021-10-02            0        661          1339  49.37
4 2021-10-02            1        650          1339  48.54
5 2021-10-02            2         28          1339   2.09
6 2021-10-03            0        648          1242  52.17
7 2021-10-03            1        563          1242  45.33
8 2021-10-03            2         31          1242   2.50
9 2021-10-04            0        601          1174  51.19


In [57]:
from itertools import product

# Tutti i giorni e categorie presenti nel dataset
tutti_giorni = transactions['date'].unique()
tutte_categorie = transactions['category_id'].unique()

print(f'Giorni totali: {len(tutti_giorni)}')
print(f'Categorie totali: {tutte_categorie}')

# Vendite reali per giorno e categoria
vendite_reali = (transactions.groupby(['date', 'category_id'])['price']
                 .count()
                 .reset_index(name='nb_ventes'))

# Tutte le combinazioni attese
attese = pd.DataFrame(list(product(tutti_giorni, tutte_categorie)), 
                      columns=['date', 'category_id'])

# Trova combinazioni mancanti
mancanti = attese.merge(vendite_reali, 
                        on=['date', 'category_id'], 
                        how='left')

mancanti = mancanti[mancanti['nb_ventes'].isna()]

print(f'\nGiorni senza vendite per categoria: {len(mancanti)}')
print(mancanti.sort_values('date'))

Giorni totali: 730
Categorie totali: [0 1 2]

Giorni senza vendite per categoria: 0
Empty DataFrame
Columns: [date, category_id, nb_ventes]
Index: []


In [58]:
# Vendite per giorno e categoria
vente_jour_categ = (transactions.groupby(['date', 'category_id'])['price']
                    .count()
                    .reset_index(name='nb_ventes'))

# Pivot table
table_vente_categ = vente_jour_categ.pivot_table(
    index='date', 
    columns='category_id', 
    values='nb_ventes', 
    aggfunc='sum'
).reset_index()

# Rinomina colonne
table_vente_categ.columns = ['date', 'cat_0', 'cat_1', 'cat_2']

# Verifica giorni con 0 vendite per categoria
table_vente_categ['jour_sans_vente'] = table_vente_categ['cat_0'].isna()
table_vente_categ

,date,cat_0,cat_1,cat_2,jour_sans_vente
0,2021-03-01,581,335,46,False
1,2021-03-02,620,276,43,False
2,2021-03-03,591,280,40,False
3,2021-03-04,563,297,43,False
4,2021-03-05,561,331,51,False
...,...,...,...,...,...
725,2023-02-24,576,271,47,False
726,2023-02-25,458,401,46,False
727,2023-02-26,508,329,53,False
728,2023-02-27,703,332,75,False


In [59]:

# 1. Sciogli le colonne delle categorie in formato lungo
df_melted = table_vente_categ.melt(
    id_vars='date',
    value_vars=['cat_0', 'cat_1', 'cat_2'],
    var_name='Categorie',
    value_name='Nb de ventes'
)

# 2. Grafico con Plotly Express
fig = px.line(
    df_melted,
    x='date',
    y='Nb de ventes',
    color='Categorie',
    title='Evolution des ventes pour chaque catégorie du 27/09 au 1/11/2021',
    labels={'date': '', 'Nb de ventes': 'Nb de ventes'}
)

# 3. Personalizzazioni come nel matplotlib
fig.update_layout(
    yaxis=dict(
        tickmode='array',
        tickvals=[0, 200, 400, 600, 800, 1000]  # stessi yticks
    ),
    xaxis=dict(
        range=[pd.Timestamp('2021-09-27'), pd.Timestamp('2021-11-01')],  # limite asse x
        tickformat='%Y-%m-%d',  # formato date
        tickangle=-45            # rotazione etichette come plt.xticks(rotation=45)
    ),
    legend_title='',              # toglie il titolo della legenda (opzionale)
    template='simple_white',
    width=1000,                   # equivalente a figsize largo (puoi regolare)
    height=500
)

# 4. Griglia solo sull'asse y
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')

fig.show()

In [60]:

# Reshape dei dati per Plotly Express (formato long)
df_long = table_vente_categ.melt(
    id_vars=['date'], 
    value_vars=['cat_0', 'cat_1', 'cat_2'],
    var_name='Categorie', 
    value_name='Nb de ventes'
)

# Rinominare le categorie per una legenda più pulita
df_long['Categorie'] = df_long['Categorie'].str.replace('cat_', 'Catégorie ')

# Creazione del grafico
fig = px.line(
    df_long, 
    x='date', 
    y='Nb de ventes', 
    color='Categorie',
    title='Evolution des ventes pour chaque catégorie — Mars 2021 à Février 2023',
    labels={'date': 'Date', 'Nb de ventes': 'Nb de ventes'},
    template='plotly_white'
)

# Personalizzazione
fig.update_layout(
    width=1200,
    height=400,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    xaxis=dict(
        tickangle=45,
        tickmode='linear',
        dtick='M1',
        tickformat='%Y-%m-%d'
    ),
    yaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=200,
        range=[0, 1050]
    )
)

fig.update_yaxes(showgrid=True, gridcolor='lightgray')
fig.show()

In [61]:
#stabilito che nel mese di ottobre 2021 la Cat 0 e 1 sono in calo nelle vendite, fare grafico a donut per vedere differenza di percentuale di vendite per il mese di ottbre 2021 e un graifoc donut per la totalità del periodo

# % vendite per categoria nel mese di ottobre 2021
oct_2021 = transactions[(transactions['year'] == 2021) & (transactions['month_num'] == 10)]

# % vendite per categoria per tutto il periodo
all_time = transactions.groupby('category_id').size()

# Calcolare la deviazione tra i 2 in generale
deviazione = (oct_2021.groupby('category_id').size() / all_time) * 100

#grafico a donut per ottobre 2021
fig_oct_2021 = px.pie(oct_2021.groupby('category_id').size().reset_index(name='vendite'), values='vendite', names='category_id', title='Répartition des ventes par catégorie - Octobre 2021')
#grafico a donut per tutto il periodo
fig_all_time = px.pie(all_time.reset_index(name='vendite'), values='vendite', names='category_id', title='Répartition des ventes par catégorie - Période totale')

fig_oct_2021.update_traces(hole=.4, hoverinfo="label+percent+name")
fig_all_time.update_traces(hole=.4, hoverinfo="label+percent+name")

fig_oct_2021.show()
fig_all_time.show()

<div style="background-color: RGB(0,150,250);" >
<h3 style="margin: auto; padding: 20px; color:#fff; ">2. Quantité vendues</h3>
</div>

<div style="background-color: RGB(255,0,0);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">A - nombre de clients par mois</h4>
</div>

In [62]:
df_panier.head(5)

,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
0,1,11.99,1,329,56,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
1,2,19.37,1,664,63,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
2,3,14.67,3,580,35,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
3,4,20.75,2,7912,34,25-34 Jeunes professionnels,2021-03-01,1,9,March,2021,3,2021-S09
4,5,16.49,1,2033,67,65+ Retraités,2021-03-01,1,9,March,2021,3,2021-S09


In [63]:
#nombre de clients par mois
df_clients_mois = transactions.groupby([transactions['date'].dt.to_period('M')]).agg(
    n_client=('client_id','count')
).reset_index().sort_values(by=['date'], ascending=True).reset_index(drop=True)

df_clients_mois.head(30)

,date,n_client
0,2021-03,28601
1,2021-04,28443
2,2021-05,28285
3,2021-06,26850
4,2021-07,24738
5,2021-08,25650
6,2021-09,33314
7,2021-10,30022
8,2021-11,28311
9,2021-12,32457


In [64]:
df_clients_mois['date'] = df_clients_mois['date'].dt.to_timestamp()


In [65]:
fig = px.line(
    df_clients_mois, 
    x='date', 
    y='n_client', 
    title='Nombre de clients par mois',
    markers=True 
)

fig.show()

In [66]:
transactions.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638


In [67]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Nombre de transactions et clients uniques par jour
daily_metrics = (transactions.groupby('date')
                 .agg(
                     nb_transactions=('session_id', 'count'),  # ou 'price', 'count'
                     nb_clients=('client_id', 'nunique')  # clients uniques
                 )
                 .reset_index())

# Calcul des métriques dérivées
daily_metrics['transactions_par_client'] = daily_metrics['nb_transactions'] / daily_metrics['nb_clients']
daily_metrics['mm_transactions'] = daily_metrics['nb_transactions'].rolling(window=7).mean()
daily_metrics['mm_clients'] = daily_metrics['nb_clients'].rolling(window=7).mean()

# Visualisation côte à côte
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Transactions vs Clients (quotidien)',
        'Transactions par client',
        'Transactions (moyenne mobile 7j)',
        'Clients (moyenne mobile 7j)'
    )
)

# Graphique 1: Transactions vs Clients
fig.add_trace(
    go.Scatter(x=daily_metrics['date'], y=daily_metrics['nb_transactions'], 
               name='Transactions', line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=daily_metrics['date'], y=daily_metrics['nb_clients'], 
               name='Clients', line=dict(color='red')),
    row=1, col=1
)

# Graphique 2: Transactions par client
fig.add_trace(
    go.Scatter(x=daily_metrics['date'], y=daily_metrics['transactions_par_client'],
               name='Trans/Client', line=dict(color='green')),
    row=1, col=2
)

# Graphique 3: Moyenne mobile transactions
fig.add_trace(
    go.Scatter(x=daily_metrics['date'], y=daily_metrics['mm_transactions'],
               name='MM Transactions', line=dict(color='blue')),
    row=2, col=1
)

# Graphique 4: Moyenne mobile clients
fig.add_trace(
    go.Scatter(x=daily_metrics['date'], y=daily_metrics['mm_clients'],
               name='MM Clients', line=dict(color='red')),
    row=2, col=2
)

fig.update_layout(height=800, width=1200, title_text="Relation Transactions vs Clients")
fig.show()

In [68]:
# Matrice de corrélation
correlation_matrix = daily_metrics[['nb_transactions', 'nb_clients', 'transactions_par_client']].corr()

# Heatmap de corrélation
fig = px.imshow(
    correlation_matrix,
    text_auto=True,
    title="Matrice de corrélation",
    labels=dict(color="Corrélation"),
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1
)
fig.show()

# Scatter plot pour visualiser la relation
fig = px.scatter(
    daily_metrics,
    x='nb_clients',
    y='nb_transactions',
    hover_data=['date'],
    title="Relation Transactions vs Clients (par jour)",
    labels={'nb_clients': 'Nombre de clients', 'nb_transactions': 'Nombre de transactions'},
)
fig.show()

<div style="background-color: RGB(255,0,0);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">B - Transactions e transactions par mois</h4>
</div>

In [69]:
#nombre de transactions en generale
print('Nombre de transactions:', len(transactions))
#nombre de transaction par mois
df_transactions_mois = transactions.groupby([transactions['date'].dt.to_period('M')]).size().reset_index(name='count')
df_transactions_mois.head(5)


Nombre de transactions: 687534


,date,count
0,2021-03,28601
1,2021-04,28443
2,2021-05,28285
3,2021-06,26850
4,2021-07,24738


In [70]:
#conversion data in timestamp
df_transactions_mois['date'] = df_transactions_mois['date'].dt.to_timestamp()

In [71]:
moyenne = df_transactions_mois['count'].mean()


fig_nb_achats_tot = px.line(
    df_transactions_mois, 
    x='date', 
    y='count', 
    title='Nombre de transactions par mois',
    markers=True 
)

fig_nb_achats_tot.add_hline(
    y=moyenne,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Moyenne: {round(moyenne, 1)}',
    annotation_position='top right'
)

fig_nb_achats_tot.show()

In [72]:
#nb achat par mois b2b
df_transactions_mois_b2b = df_transactions_b2b.groupby([df_transactions_b2b['date'].dt.to_period('M')]).size().reset_index(name='count')
df_transactions_mois_b2b.head(5)

,date,count
0,2021-03,1954
1,2021-04,1973
2,2021-05,2013
3,2021-06,1812
4,2021-07,1663


In [73]:
#conversion date en timestamp
df_transactions_mois_b2b['date'] = df_transactions_mois_b2b['date'].dt.to_timestamp()

In [74]:
fig_b2b_nb_achats = px.line(
    df_transactions_mois_b2b, 
    x='date', 
    y='count', 
    title='Nombre de transactions B2B par mois',
    markers=True 
)
fig_b2b_nb_achats.show()


In [75]:
moyenne = df_transactions_mois_b2b['count'].mean() 

fig_b2b_nb_achats = px.line(
    df_transactions_mois_b2b, 
    x='date', 
    y='count', 
    title='Nombre de transactions B2B par mois',
    markers=True 
)

fig_b2b_nb_achats.add_hline(
    y=moyenne,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Moyenne: {round(moyenne, 1)}',
    annotation_position='top right'
)

fig_b2b_nb_achats.show()

In [76]:
#calcul moyen d'achat par mois pour b2b
df_transactions_mois_b2b['avg_amount'] = df_transactions_mois_b2b['count'] / df_transactions_mois_b2b['count'].sum()
print(df_transactions_mois_b2b.head(5))

        date  count  avg_amount
0 2021-03-01   1954    0.041752
1 2021-04-01   1973    0.042158
2 2021-05-01   2013    0.043013
3 2021-06-01   1812    0.038718
4 2021-07-01   1663    0.035534


<div style="background-color: RGB(255,0,0);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">C - Nombre de produits vendus, Top et Flops</h4>
</div>

In [77]:
#nombre de produits vendus
transac_totale = df_transactions_mois['count'].sum()
print(f"Le nombre total de produits vendus est de : {transac_totale} produits.")


Le nombre total de produits vendus est de : 687534 produits.


In [78]:
#Top Vendus
top_produits = (transactions.groupby('id_prod')['price']
                .sum()
                .reset_index()
                .rename(columns={'price': 'ca'})
                .sort_values('ca', ascending=False))

# Top 10
print(top_produits.head(10))

# Flop 10
print(top_produits.tail(10))

     id_prod        ca
3096   2_159  94893.50
3070   2_135  69334.95
3045   2_112  65407.76
3034   2_102  60736.78
3152   2_209  56971.86
2619   1_395  56617.47
2591   1_369  56136.60
3043   2_110  53846.25
2607   1_383  53834.43
2641   1_414  53522.18
     id_prod    ca
925   0_1840  2.56
2179   0_898  2.54
549   0_1498  2.48
802   0_1728  2.27
2079   0_807  1.99
665   0_1601  1.99
1784   0_541  1.99
719   0_1653  1.98
313   0_1284  1.38
595   0_1539  0.99


In [79]:
# top produits b2b
#Top Vendus
top_produits = (df_transactions_b2b.groupby('id_prod')['price']
                .sum()
                .reset_index()
                .rename(columns={'price': 'ca'})
                .sort_values('ca', ascending=False))

# Top 10
print(top_produits.head(10))

# Flop 10
print(top_produits.tail(10))

     id_prod       ca
2190   2_112  6621.86
2281   2_209  6229.11
2188   2_110  5976.00
2207   2_135  5864.15
2320    2_39  5567.04
2318    2_37  5339.91
2181   2_102  5086.04
2280   2_208  4883.43
2186   2_109  4859.10
2241   2_167  4739.80
     id_prod    ca
457   0_1510  2.18
1215    0_32  2.14
1127  0_2288  2.06
756   0_1871  1.99
1405   0_587  1.99
165   0_1191  1.98
153    0_118  1.98
12    0_1010  1.96
812   0_1935  0.99
1357   0_528  0.62


In [80]:
#Top/flop per anno
top_produits_year = (transactions.groupby(['year', 'id_prod'])['price']
                     .sum()
                     .reset_index()
                     .rename(columns={'price': 'ca'})
                     .sort_values(['year', 'ca'], ascending=[True, False]))

# Top 10 per anno
for anno in top_produits_year['year'].unique():
    print(f'--- {anno} ---')
    print(top_produits_year[top_produits_year['year'] == anno].head(10))
    print()

--- 2021 ---
      year id_prod        ca
3086  2021   2_159  39563.29
3060  2021   2_135  27389.03
3036  2021   2_112  26149.59
3025  2021   2_102  23242.02
3034  2021   2_110  23219.25
2610  2021   1_395  22786.14
3142  2021   2_209  22746.75
2582  2021   1_369  22430.65
2598  2021   1_383  22235.33
2632  2021   1_414  20994.23

--- 2022 ---
      year id_prod        ca
6309  2022   2_159  47592.74
6283  2022   2_135  36633.69
6259  2022   2_112  32771.45
6248  2022   2_102  31403.34
6365  2022   2_209  29395.80
5834  2022   1_395  29163.94
5806  2022   1_369  28811.99
6412  2022    2_39  27951.18
5856  2022   1_414  27690.46
5822  2022   1_383  27366.56

--- 2023 ---
      year id_prod       ca
9120  2023   2_159  7737.47
9076  2023   2_112  6486.72
9067  2023   2_102  6091.42
9094  2023   2_135  5312.23
9128  2023   2_166  5060.88
9119  2023   2_158  4944.40
9214  2023    2_39  4929.15
8657  2023   1_369  4893.96
8702  2023   1_414  4837.49
9173  2023   2_209  4829.31



In [81]:
#flop produit par année
flop_produits_year = (transactions.groupby(['year', 'id_prod'])['price']
                     .sum()
                     .reset_index()
                     .rename(columns={'price': 'ca'})
                     .sort_values(['year', 'ca'], ascending=[True, True]))

# Flop 10 per anno
for annee in flop_produits_year['year'].unique():
    print(f'--- {annee} ---')
    print(flop_produits_year[flop_produits_year['year'] == annee].head(10))
    print()

--- 2021 ---
      year id_prod    ca
609   2021  0_1552  0.98
594   2021  0_1539  0.99
921   2021  0_1840  1.28
313   2021  0_1284  1.38
1856  2021   0_612  1.76
940   2021  0_1858  1.83
210   2021  0_1191  1.98
714   2021  0_1651  1.98
832   2021  0_1759  1.98
1890  2021   0_643  1.98

--- 2022 ---
      year id_prod    ca
4163  2022  0_1844  0.77
5402  2022   0_902  0.92
3438  2022  0_1171  0.99
3459  2022  0_1191  0.99
3811  2022  0_1511  0.99
4073  2022  0_1759  0.99
5121  2022   0_643  0.99
4996  2022   0_528  1.24
4038  2022  0_1726  1.57
5470  2022   0_964  1.61

--- 2023 ---
      year id_prod    ca
7270  2023  0_1844  0.77
6662  2023  0_1203  0.99
6962  2023  0_1511  0.99
7095  2023  0_1651  0.99
7203  2023  0_1771  0.99
6553  2023  0_1088  1.09
7000  2023  0_1550  1.21
7573  2023  0_2144  1.25
7267  2023  0_1840  1.28
6658  2023   0_120  1.32



In [188]:
# Top prodotti per anno (in base al numero di vendite)
top_produits_year = (transactions.groupby(['year', 'id_prod'])
                     .size()
                     .reset_index(name='nb_ventes')
                     .sort_values(['year', 'nb_ventes'], ascending=[True, False])) # Ordine decrescente sulle vendite

# Top 10 per anno
for annee in top_produits_year['year'].unique():
    print(f'--- Top 10 {annee} ---')
    print(top_produits_year[top_produits_year['year'] == annee].head(10))
    print()

--- Top 10 2021 ---
      year id_prod  nb_ventes
2582  2021   1_369        935
2635  2021   1_417        912
2632  2021   1_414        881
2724  2021   1_498        879
2644  2021   1_425        853
2631  2021   1_413        839
2611  2021   1_396        817
2613  2021   1_398        816
2624  2021   1_407        815
2620  2021   1_403        811

--- Top 10 2022 ---
      year id_prod  nb_ventes
5806  2022   1_369       1201
5859  2022   1_417       1197
5856  2022   1_414       1162
5948  2022   1_498       1161
5868  2022   1_425       1132
5844  2022   1_403       1071
5855  2022   1_413       1035
5854  2022   1_412       1032
5847  2022   1_406       1031
5835  2022   1_396       1025

--- Top 10 2023 ---
      year id_prod  nb_ventes
8657  2023   1_369        204
8702  2023   1_414        203
8700  2023   1_412        179
8709  2023   1_425        178
8695  2023   1_406        177
8670  2023   1_381        176
8685  2023   1_397        172
8696  2023   1_407        163
8701  20

In [186]:
# Flop prodotti per anno (in base al numero di vendite)
flop_produits_year = (transactions.groupby(['year', 'id_prod'])
                      .size() # Conteggia il numero di righe/vendite per ciascun prodotto
                      .reset_index(name='nb_ventes') # Assegna il nome alla nuova colonna
                      .sort_values(['year', 'nb_ventes'], ascending=[True, True]))

# Flop 10 per anno
for annee in flop_produits_year['year'].unique():
    print(f'--- {annee} ---')
    print(flop_produits_year[flop_produits_year['year'] == annee].head(10))
    print()

--- 2021 ---
     year id_prod  nb_ventes
81   2021  0_1074          1
112  2021  0_1101          1
128  2021  0_1116          1
132  2021  0_1120          1
166  2021  0_1151          1
181  2021  0_1165          1
255  2021  0_1231          1
257  2021  0_1233          1
299  2021  0_1271          1
307  2021  0_1279          1

--- 2022 ---
      year id_prod  nb_ventes
3269  2022  0_1012          1
3288  2022  0_1032          1
3293  2022  0_1037          1
3380  2022  0_1116          1
3406  2022  0_1141          1
3438  2022  0_1171          1
3459  2022  0_1191          1
3505  2022  0_1233          1
3509  2022  0_1237          1
3555  2022  0_1279          1

--- 2023 ---
      year id_prod  nb_ventes
6476  2023   0_100          1
6482  2023  0_1005          1
6483  2023  0_1006          1
6488  2023  0_1010          1
6494  2023  0_1019          1
6495  2023   0_102          1
6506  2023  0_1032          1
6530  2023   0_106          1
6551  2023  0_1084          1
6553  2023

In [82]:
#top/flop per categoria totale


In [83]:
#top/lop per categoria per anno

In [84]:
transactions.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638


In [85]:

# Numero di vendite per giorno e categoria
nv_jour_cat = (transactions.groupby(['date', 'category_id'])['price']
               .count()  # <-- count invece di sum
               .reset_index(name='nb_ventes'))

# Pivot table
nv_pivot = nv_jour_cat.pivot_table(
    index='date',
    columns='category_id',
    values='nb_ventes',
    aggfunc='sum'
).reset_index()

nv_pivot.columns = ['date', 'cat_0', 'cat_1', 'cat_2']

# Moyenne mobile 7 jours
nv_pivot['mm_cat_0'] = nv_pivot['cat_0'].rolling(window=7).mean()
nv_pivot['mm_cat_1'] = nv_pivot['cat_1'].rolling(window=7).mean()
nv_pivot['mm_cat_2'] = nv_pivot['cat_2'].rolling(window=7).mean()

# Grafico Plotly
fig = px.line(
    nv_pivot,
    x='date',
    y=['mm_cat_0', 'mm_cat_1', 'mm_cat_2'],
    labels={
        'value': 'Nombre de ventes',
        'date': 'Date',
        'variable': 'Catégorie'
    },
    title='Evolution du nombre de ventes par catégorie — moyenne mobile 7 jours'
)

fig.update_layout(
    width=1200,
    height=600,
    legend_title_text='Catégories',
    xaxis_title='Date',
    yaxis_title='Nombre de ventes',
    hovermode='x unified'
)

fig.for_each_trace(lambda t: t.update(name=t.name.replace('mm_', 'Catégorie ')))
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()

<div style="background-color: RGB(255,0,0);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">E Analyse des prix</h4>
</div>

In [86]:
#analyse des prix
analyse_prix = transactions.groupby(['category_id','product_num' ]).agg(
    produit_id=('id_prod', 'first'),
    nb_ventes=('price', 'count'),
    ca_total=('price', 'sum'),
    prix=('price', 'first'),
    prix_moyen=('price','mean'),
    prix_min=('price','min'),
    prix_max=('price','max',)
).reset_index()
    
analyse_prix = analyse_prix.round(2)
analyse_prix.tail(10)
# verification pour head & tail pour voir si les prix changent ou pas - ils ne changent pas.

,category_id,product_num,produit_id,nb_ventes,ca_total,prix,prix_moyen,prix_min,prix_max
3255,2,229,2_229,61,4940.39,80.99,80.99,80.99,80.99
3256,2,230,2_230,403,22987.12,57.04,57.04,57.04,57.04
3257,2,231,2_231,75,8399.25,111.99,111.99,111.99,111.99
3258,2,232,2_232,65,6763.90,104.06,104.06,104.06,104.06
3259,2,233,2_233,51,8822.49,172.99,172.99,172.99,172.99
3260,2,234,2_234,39,3166.41,81.19,81.19,81.19,81.19
3261,2,235,2_235,42,8231.58,195.99,195.99,195.99,195.99
3262,2,236,2_236,80,9480.80,118.51,118.51,118.51,118.51
3263,2,237,2_237,193,7911.07,40.99,40.99,40.99,40.99
3264,2,238,2_238,5,469.60,93.92,93.92,93.92,93.92


<div style="background-color: RGB(255,0,0);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">D -Tranches prix CA</h4>
</div>

In [87]:
bins = [0, 10, 20, 50, 100, 500]
labels = ['0-10€', '10-20€', '20-50€', '50-100€', '100€+']

transactions['tranche_prix'] = pd.cut(transactions['price'], 
                                       bins=bins, 
                                       labels=labels, 
                                       right=False)

In [88]:
tranche_prix = transactions.groupby('tranche_prix').agg(
    nb_ventes=('price', 'count'),
    ca_total=('price', 'sum')
).reset_index()

tranche_prix['pct_ventes'] = (tranche_prix['nb_ventes'] / tranche_prix['nb_ventes'].sum() * 100).round(2)
tranche_prix['pct_ca'] = (tranche_prix['ca_total'] / tranche_prix['ca_total'].sum() * 100).round(2)

print(tranche_prix)

  tranche_prix  nb_ventes    ca_total  pct_ventes  pct_ca
0        0-10€     215399  1412282.71       31.33   11.74
1       10-20€     323506  4847815.79       47.05   40.31
2       20-50€     116833  3170333.17       16.99   26.36
3      50-100€      24863  1581748.27        3.62   13.15
4        100€+       6933  1015483.16        1.01    8.44


/tmp/ipykernel_4112/2592463392.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [89]:
# Volume vendite per tranche
fig1 = px.bar(tranche_prix, x='tranche_prix', y='nb_ventes',
              title='Nombre de ventes par tranche de prix',
              labels={'nb_ventes': 'Nb ventes', 'tranche_prix': 'Tranche de prix'},
              text='pct_ventes')
fig1.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig1.show()

# CA per tranche
fig2 = px.bar(tranche_prix, x='tranche_prix', y='ca_total',
              title='CA par tranche de prix',
              labels={'ca_total': 'CA (€)', 'tranche_prix': 'Tranche de prix'},
              text='pct_ca')
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.show()

<div style="background-color: RGB(0,150,250);" >
<h3 style="margin: auto; padding: 20px; color:#fff; ">3. Profils clients</h3>
</div>

<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">A - CA pour clients BtoB</h4>
</div>

In [90]:
#Ca b2b
ca_b2b = df_transactions_b2b['price'].sum()
print(f"Le chiffre d'affaires total des ventes B2B est de : {ca_b2b:.2f} euros. Correspond au {((ca_b2b/ca_total)*100):.2f}% du total")


Le chiffre d'affaires total des ventes B2B est de : 884296.09 euros. Correspond au 7.35% du total


In [91]:
# b2c achats
transactions_b2c = transactions[~transactions['client_id'].isin(liste_b2b)]
# panier b2c
df_panier_b2c = df_panier[df_panier['client_id'].isin(transactions_b2c['client_id'].tolist())]

# Achats par client B2C
achats_par_client_b2c = (transactions_b2c.groupby('client_id')['id_prod']
                          .count()
                          .reset_index(name='nb_achats')
                          .sort_values('nb_achats', ascending=False))
df_panier_b2c.head()



,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
0,1,11.99,1,329,56,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
1,2,19.37,1,664,63,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
2,3,14.67,3,580,35,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
3,4,20.75,2,7912,34,25-34 Jeunes professionnels,2021-03-01,1,9,March,2021,3,2021-S09
4,5,16.49,1,2033,67,65+ Retraités,2021-03-01,1,9,March,2021,3,2021-S09


In [92]:
transactions_b2c.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num,tranche_prix
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259,10-20€
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390,10-20€
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352,0-10€
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110,0-10€
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638,0-10€


In [93]:
#probabilemente un grafico inutile

fig_achats_b2c = px.histogram(achats_par_client_b2c, x='nb_achats', 
                   title='Distribution des achats par B2C',
                   labels={'nb_achats': 'Nb achats'})
fig_achats_b2c.show() # modificare gli assi perché tutto sia visibile

In [94]:
#distribution d'acaht B2B
transactions_b2b = transactions[transactions['client_id'].isin(liste_b2b)]

# Achats par client B2B
achats_par_client_b2b = (transactions_b2b.groupby('client_id')['id_prod']
                          .count()
                          .reset_index(name='nb_achats')
                          .sort_values('nb_achats', ascending=False))

In [95]:
fig_achats_b2b = px.histogram(achats_par_client_b2b, x='nb_achats', 
                   title='Distribution des achats par B2B',
                   labels={'nb_achats': 'Nb achats'})
fig_achats_b2b.show() 

In [96]:
print('CA totale con B2B:', round(transactions['price'].sum(), 2))
print('CA totale senza B2B:', round(transactions_b2c['price'].sum(), 2))
print('CA generato dai B2B:', round(transactions_b2b['price'].sum(), 2))
print('% CA B2B:', round(transactions_b2b['price'].sum() / transactions['price'].sum() * 100, 2), '%')

CA totale con B2B: 12027663.1
CA totale senza B2B: 11143367.01
CA generato dai B2B: 884296.09
% CA B2B: 7.35 %


<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">B - Courbe Lorenz</h4>
</div>

In [97]:
transactions_b2c.head()

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num,tranche_prix
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259,10-20€
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390,10-20€
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352,0-10€
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110,0-10€
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638,0-10€


In [98]:
ca_clients = transactions.groupby('client_id')['price'].sum().reset_index(name='ca')
ca_clients = ca_clients.sort_values('ca').reset_index(drop=True)

In [99]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# --- Supponiamo che ca_clients sia già stato creato ---
ca_clients = transactions.groupby('client_id')['price'].sum().reset_index(name='ca')
ca_clients = ca_clients.sort_values('ca').reset_index(drop=True)
ca_clients['ca_cumule'] = ca_clients['ca'].cumsum()
ca_clients['ca_cumule_pct'] = ca_clients['ca_cumule'] / ca_clients['ca'].sum() * 100
ca_clients['clients_cumule_pct'] = (np.arange(1, len(ca_clients) + 1) / len(ca_clients)) * 100

# 1. Courbe de Lorenz (titre et labels en français)
fig = px.line(
    ca_clients, 
    x='clients_cumule_pct', 
    y='ca_cumule_pct',
    title='Courbe de Lorenz – Concentration du chiffre d’affaires parmi les clients',
    labels={
        'clients_cumule_pct': '% cumulé des clients', 
        'ca_cumule_pct': '% cumulé du CA'
    }
)

# Rendre la courbe plus épaisse pour une meilleure lisibilité
fig.update_traces(line=dict(color='royalblue', width=3))

# 2. Droite d'égalité parfaite (légende en français)
fig.add_trace(
    go.Scatter(
        x=[0, 100], y=[0, 100],
        mode='lines',
        line=dict(dash='dash', color='grey', width=2),
        name='Égalité parfaite'  # traduit
    )
)

# 3. Mise en page adaptée
fig.update_layout(
    xaxis_range=[0, 100],
    yaxis_range=[0, 100],
    template='simple_white',
    xaxis_title_font=dict(size=12),
    yaxis_title_font=dict(size=12),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.7)')
)

# 4. Calcul et affichage de l'indice de Gini (annotation en français)
y_vals = ca_clients['ca'].values
n = len(y_vals)
gini = (2 * np.sum(np.arange(1, n+1) * y_vals)) / (n * y_vals.sum()) - (n+1)/n

fig.add_annotation(
    x=80, y=20,
    text=f'Indice de Gini = {gini:.3f}',  # traduit
    showarrow=False,
    font=dict(size=14, color='darkblue'),
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='black',
    borderwidth=1
)

fig.show()

In [100]:


# --- Dati: CA per cliente già ordinati in modo decrescente ---
# ca_clients = transactions.groupby('client_id')['price'].sum().reset_index(name='ca')
# ca_clients = ca_clients.sort_values('ca', ascending=False).reset_index(drop=True)

# 1. Calcoli per il Pareto
n = len(ca_clients)
ca_clients['ca_cumulato'] = ca_clients['ca'].cumsum()
ca_totale = ca_clients['ca'].sum()
ca_clients['ca_cumulato_pct'] = ca_clients['ca_cumulato'] / ca_totale * 100
ca_clients['clienti_cumulati_pct'] = (np.arange(1, n + 1) / n) * 100
ca_clients['clienti_cumulati'] = np.arange(1, n + 1)

# 2. Trovare il punto dove si supera l'80%
soglia_80 = 80
punto_soglia = ca_clients[ca_clients['ca_cumulato_pct'] >= soglia_80].iloc[0]
clienti_necessari = punto_soglia['clienti_cumulati']
clienti_percentuale = punto_soglia['clienti_cumulati_pct']
ca_soglia = punto_soglia['ca_cumulato_pct']

print(f"Per raggiungere l'80% del CA servono:")
print(f"- {clienti_necessari:.0f} clienti su {n} totali")
print(f"- {clienti_percentuale:.1f}% dei clienti")
print(f"- Contribuiscono al {ca_soglia:.1f}% del CA totale")

# 3. Grafico Pareto
fig = go.Figure()

# Barre: CA per singolo cliente
fig.add_trace(go.Bar(
    x=ca_clients['clienti_cumulati'],
    y=ca_clients['ca'],
    name='CA per cliente',
    marker_color='steelblue',
    opacity=0.7
))

# Linea: CA cumulato percentuale
fig.add_trace(go.Scatter(
    x=ca_clients['clienti_cumulati'],
    y=ca_clients['ca_cumulato_pct'],
    name='% CA cumulato',
    yaxis='y2',
    line=dict(color='red', width=2)
))

# Linea orizzontale dell'80%
fig.add_hline(y=80, line_dash="dash", line_color="green", 
              annotation_text="80%", annotation_position="top right")

# Linea verticale al punto di soglia
fig.add_vline(x=clienti_necessari, line_dash="dash", line_color="orange",
              annotation_text=f"{clienti_necessari:.0f} clienti ({clienti_percentuale:.1f}%)", 
              annotation_position="top")

# Layout con doppio asse
fig.update_layout(
    title=f'Analisi Pareto - {clienti_necessari:.0f} clienti ({clienti_percentuale:.1f}%) generano l\'80% del CA',
    xaxis_title='Clienti ordinati per CA (decrescente)',
    yaxis_title='CA per cliente (€)',
    yaxis2=dict(
        title='% CA cumulato',
        overlaying='y',
        side='right',
        range=[0, 105]
    ),
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    height=500
)

fig.show()

# 4. Tabella riassuntiva per i primi clienti
print("\n--- TOP 10 clienti ---")
top_n = min(10, n)
tabella = ca_clients[['clienti_cumulati', 'ca', 'ca_cumulato_pct']].head(top_n)
tabella.columns = ['Cliente #', 'CA (€)', '% CA Cumulato']
print(tabella.round(2))

Per raggiungere l'80% del CA servono:
- 8208 clienti su 8600 totali
- 95.4% dei clienti
- Contribuiscono al 80.0% del CA totale



--- TOP 10 clienti ---
   Cliente #  CA (€)  % CA Cumulato
0          1    6.31            0.0
1          2    8.30            0.0
2          3    9.98            0.0
3          4   11.20            0.0
4          5   13.36            0.0
5          6   15.72            0.0
6          7   15.98            0.0
7          8   16.07            0.0
8          9   17.64            0.0
9         10   17.89            0.0


In [101]:

# --- Dati: CA per cliente già ordinati in modo decrescente ---
# ca_clients = transactions.groupby('client_id')['price'].sum().reset_index(name='ca')
# ca_clients = ca_clients.sort_values('ca', ascending=False).reset_index(drop=True)

# 1. Calcoli per il Pareto
n = len(ca_clients)
ca_clients['ca_cumulato'] = ca_clients['ca'].cumsum()
ca_totale = ca_clients['ca'].sum()
ca_clients['ca_cumulato_pct'] = ca_clients['ca_cumulato'] / ca_totale * 100
ca_clients['clienti_cumulati_pct'] = (np.arange(1, n + 1) / n) * 100
ca_clients['clienti_cumulati'] = np.arange(1, n + 1)

# 2. Trovare il punto dove si supera l'80%
soglia_80 = 80
punto_soglia = ca_clients[ca_clients['ca_cumulato_pct'] >= soglia_80].iloc[0]
clienti_necessari = punto_soglia['clienti_cumulati']
clienti_percentuale = punto_soglia['clienti_cumulati_pct']
ca_soglia = punto_soglia['ca_cumulato_pct']

print(f"Afin d'arriver au 80% du CA servent:")
print(f"- {clienti_necessari:.0f} clients su {n} totali")
print(f"- {clienti_percentuale:.1f}% dei clients")
print(f"- Contribuent au {ca_soglia:.1f}% du CA totale")

# 3. Grafico Pareto
fig = go.Figure()

# Barre: CA per singolo cliente
fig.add_trace(go.Bar(
    x=ca_clients['clienti_cumulati'],
    y=ca_clients['ca'],
    name='CA per cliente',
    marker_color='steelblue',
    opacity=0.7
))

# Linea: CA cumulato percentuale
fig.add_trace(go.Scatter(
    x=ca_clients['clienti_cumulati'],
    y=ca_clients['ca_cumulato_pct'],
    name='% CA cumulato',
    yaxis='y2',
    line=dict(color='red', width=2)
))

# Linea orizzontale dell'80%
fig.add_hline(y=80, line_dash="dash", line_color="green", 
              annotation_text="80%", annotation_position="top right")

# Linea verticale al punto di soglia
fig.add_vline(x=clienti_necessari, line_dash="dash", line_color="orange",
              annotation_text=f"{clienti_necessari:.0f} clients ({clienti_percentuale:.1f}%)", 
              annotation_position="top")

# Layout con doppio asse
fig.update_layout(
    title=f'Analyse Pareto - {clienti_necessari:.0f} clients ({clienti_percentuale:.1f}%) generent l\'80% du CA',
    xaxis_title='Clients ordonnes par CA (décroissant)',
    yaxis_title='CA par client (€)',
    yaxis2=dict(
        title='% CA cumulé',
        overlaying='y',
        side='right',
        range=[0, 105]
    ),
    template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    height=500
)

fig.show()

# 4. Tabella riassuntiva per i primi clienti (con client_id)
print("\n--- TOP 10 clients ---")
top_n = min(10, n)
tabella = ca_clients[['client_id', 'clienti_cumulati', 'ca', 'ca_cumulato_pct']].tail(top_n)
tabella.columns = ['ID Cliente', 'Posizione', 'CA (€)', '% CA Cumulato']
print(tabella.round(2))

# 5. Tabella dei clienti che superano l'80% (opzionale)
print(f"\n--- Clients qui contribuent au 80% du CA (premier {clienti_necessari:.0f} positions) ---")
clienti_80 = ca_clients[['client_id', 'clienti_cumulati', 'ca', 'ca_cumulato_pct']].head(int(clienti_necessari))
print(f"Totale: {len(clienti_80)} clients")
print(f"CA cumulé: {clienti_80['ca_cumulato_pct'].iloc[-1]:.1f}%")
print("\nPremière 5 lignes:")
print(clienti_80.head().round(2))

Afin d'arriver au 80% du CA servent:
- 8208 clients su 8600 totali
- 95.4% dei clients
- Contribuent au 80.0% du CA totale



--- TOP 10 clients ---
      ID Cliente  Posizione     CA (€)  % CA Cumulato
8590        7959       8591    5135.75          92.43
8591        7319       8592    5155.77          92.47
8592        2899       8593    5214.05          92.52
8593        2140       8594    5260.18          92.56
8594        3263       8595    5276.87          92.60
8595        1570       8596    5285.82          92.65
8596        3454       8597  114110.57          93.60
8597        6714       8598  153918.60          94.88
8598        4958       8599  290227.03          97.29
8599        1609       8600  326039.89         100.00

--- Clients qui contribuent au 80% du CA (premier 8208 positions) ---
Totale: 8208 clients
CA cumulé: 80.0%

Première 5 lignes:
   client_id  clienti_cumulati     ca  ca_cumulato_pct
0       8351                 1   6.31              0.0
1       8140                 2   8.30              0.0
2       8114                 3   9.98              0.0
3       4648                 4  1

In [102]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Dati: CA per cliente già ordinati in modo crescente per la Lorenz ---
# ca_clients = transactions.groupby('client_id')['price'].sum().reset_index(name='ca')
# ca_clients = ca_clients.sort_values('ca', ascending=True).reset_index(drop=True)  # CRESCENTE per Lorenz

# 1. Calcoli per la Curva di Lorenz
n = len(ca_clients)
ca_clients['ca_cumulato'] = ca_clients['ca'].cumsum()
ca_totale = ca_clients['ca'].sum()
ca_clients['ca_cumulato_pct'] = ca_clients['ca_cumulato'] / ca_totale * 100
ca_clients['clienti_cumulati_pct'] = (np.arange(1, n + 1) / n) * 100

# Aggiungo il punto (0,0) per la curva
x_lorenz = [0] + ca_clients['clienti_cumulati_pct'].tolist()
y_lorenz = [0] + ca_clients['ca_cumulato_pct'].tolist()

# 2. Calcolo coefficiente di Gini
# Formula: Gini = 1 - 2 * ∫(Lorenz) 
# Metodo discreto: Gini = (2 * Σ(i * y_i)) / (n * Σ(y_i)) - (n+1)/n
# dove y_i è ordinato in modo CRESCENTE
y_vals = ca_clients['ca'].values
gini = (2 * np.sum(np.arange(1, n+1) * y_vals)) / (n * y_vals.sum()) - (n+1)/n

print(f"📊 Coefficiente di Gini: {gini:.4f}")
print(f"   - Gini = 0: perfetta uguaglianza")
print(f"   - Gini = 1: massima disuguaglianza")
print(f"   - Interpretazione: {gini:.1%} di concentrazione del CA")

# 3. Calcoli per il Pareto (ordinamento decrescente)
ca_clients_desc = ca_clients.sort_values('ca', ascending=False).reset_index(drop=True)
ca_clients_desc['ca_cumulato'] = ca_clients_desc['ca'].cumsum()
ca_clients_desc['ca_cumulato_pct'] = ca_clients_desc['ca_cumulato'] / ca_totale * 100
ca_clients_desc['clienti_cumulati'] = np.arange(1, n + 1)

# Trovare il punto dove si supera l'80%
soglia_80 = 80
punto_soglia = ca_clients_desc[ca_clients_desc['ca_cumulato_pct'] >= soglia_80].iloc[0]
clienti_necessari = punto_soglia['clienti_cumulati']
clienti_percentuale = punto_soglia['clienti_cumulati'] / n * 100
ca_soglia = punto_soglia['ca_cumulato_pct']

print(f"\n📈 Per raggiungere l'80% del CA servono:")
print(f"   - {clienti_necessari:.0f} clienti su {n} totali")
print(f"   - {clienti_percentuale:.1f}% dei clienti")
print(f"   - Contribuiscono al {ca_soglia:.1f}% del CA totale")

# 4. CREAZIONE GRAFICO: Curva di Lorenz + Pareto (subplot)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Curva di Lorenz', 'Analisi Pareto'),
    specs=[[{"secondary_y": False}, {"secondary_y": True}]]
)

# ---- GRAFICO 1: Curva di Lorenz (sinistra) ----
# Linea di equidistribuzione
fig.add_trace(
    go.Scatter(
        x=[0, 100], 
        y=[0, 100],
        mode='lines',
        line=dict(dash='dash', color='grey', width=2),
        name='Equidistribuzione'
    ),
    row=1, col=1
)

# Curva di Lorenz
fig.add_trace(
    go.Scatter(
        x=x_lorenz,
        y=y_lorenz,
        mode='lines',
        line=dict(color='blue', width=3),
        name='Curva di Lorenz',
        fill='toself',
        fillcolor='rgba(0, 0, 255, 0.1)'
    ),
    row=1, col=1
)

# Area tra le due curve (opzionale - evidenzia la disuguaglianza)
fig.add_trace(
    go.Scatter(
        x=[0, 100, 100, 0],
        y=[0, 100, 0, 0],
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.05)',
        line=dict(width=0),
        name='Area di disuguaglianza',
        showlegend=False
    ),
    row=1, col=1
)

# Aggiungo il valore di Gini sul grafico
fig.add_annotation(
    x=70, y=20,
    text=f'Gini = {gini:.4f}',
    showarrow=False,
    font=dict(size=14, color='red', weight='bold'),
    row=1, col=1
)

# ---- GRAFICO 2: Pareto (destra) ----
# Barre: CA per singolo cliente
fig.add_trace(
    go.Bar(
        x=ca_clients_desc['clienti_cumulati'],
        y=ca_clients_desc['ca'],
        name='CA per cliente',
        marker_color='steelblue',
        opacity=0.7,
        showlegend=False
    ),
    row=1, col=2
)

# Linea: CA cumulato percentuale
fig.add_trace(
    go.Scatter(
        x=ca_clients_desc['clienti_cumulati'],
        y=ca_clients_desc['ca_cumulato_pct'],
        name='% CA cumulato',
        line=dict(color='red', width=2),
        yaxis='y2'
    ),
    row=1, col=2
)

# Linea orizzontale dell'80%
fig.add_hline(
    y=80, 
    line_dash="dash", 
    line_color="green",
    line_width=2,
    annotation_text="80%",
    annotation_position="top right",
    annotation_font=dict(color="green", size=10),
    row=1, col=2
)

# Linea verticale al punto di soglia
fig.add_vline(
    x=clienti_necessari, 
    line_dash="dash", 
    line_color="orange",
    line_width=2,
    annotation_text=f"{clienti_necessari:.0f} clienti<br>({clienti_percentuale:.1f}%)",
    annotation_position="top",
    annotation_font=dict(color="orange", size=9),
    row=1, col=2
)

# ---- Layout ----
fig.update_layout(
    title_text=f'Analisi Concentrazione CA - Coeff. Gini: {gini:.4f}',
    height=500,
    width=1000,
    template='plotly_white',
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Assi del primo grafico (Lorenz)
fig.update_xaxes(
    title_text="% cumulata clienti",
    range=[0, 100],
    row=1, col=1
)
fig.update_yaxes(
    title_text="% cumulata CA",
    range=[0, 100],
    row=1, col=1
)

# Assi del secondo grafico (Pareto)
fig.update_xaxes(
    title_text="Clienti ordinati per CA",
    row=1, col=2
)
fig.update_yaxes(
    title_text="CA per cliente (€)",
    titlefont=dict(color='steelblue'),
    tickfont=dict(color='steelblue'),
    side='left',
    row=1, col=2
)
fig.update_yaxes(
    title_text="% CA cumulato",
    titlefont=dict(color='red'),
    tickfont=dict(color='red'),
    overlaying='y',
    side='right',
    range=[0, 105],
    row=1, col=2
)

fig.show()

# 5. Tabella riassuntiva
print("\n--- TOP 10 clienti ---")
top_n = min(10, n)
tabella = ca_clients_desc[['client_id', 'clienti_cumulati', 'ca', 'ca_cumulato_pct']].head(top_n)
tabella.columns = ['ID Cliente', 'Posizione', 'CA (€)', '% CA Cumulato']
print(tabella.round(2))

# 6. Statistiche aggiuntive
print("\n--- STATISTICHE DI CONCENTRAZIONE ---")
print(f"Totale clienti: {n}")
print(f"CA totale: €{ca_totale:,.2f}")
print(f"CA medio per cliente: €{ca_totale/n:,.2f}")
print(f"CA mediano: €{np.median(ca_clients['ca']):,.2f}")
print(f"CA massimo: €{ca_clients['ca'].max():,.2f}")
print(f"CA minimo: €{ca_clients['ca'].min():,.2f}")
print(f"Rapporto max/min: {ca_clients['ca'].max()/ca_clients['ca'].min():.1f}x")

📊 Coefficiente di Gini: 0.4419
   - Gini = 0: perfetta uguaglianza
   - Gini = 1: massima disuguaglianza
   - Interpretazione: 44.2% di concentrazione del CA

📈 Per raggiungere l'80% del CA servono:
   - 4330 clienti su 8600 totali
   - 50.3% dei clienti
   - Contribuiscono al 80.0% del CA totale



--- TOP 10 clienti ---
   ID Cliente  Posizione     CA (€)  % CA Cumulato
0        1609          1  326039.89           2.71
1        4958          2  290227.03           5.12
2        6714          3  153918.60           6.40
3        3454          4  114110.57           7.35
4        1570          5    5285.82           7.40
5        3263          6    5276.87           7.44
6        2140          7    5260.18           7.48
7        2899          8    5214.05           7.53
8        7319          9    5155.77           7.57
9        7959         10    5135.75           7.61

--- STATISTICHE DI CONCENTRAZIONE ---
Totale clienti: 8600
CA totale: €12,027,663.10
CA medio per cliente: €1,398.57
CA mediano: €1,045.69
CA massimo: €326,039.89
CA minimo: €6.31
Rapporto max/min: 51670.3x


In [103]:
analyse_prix.head(5)

,category_id,product_num,produit_id,nb_ventes,ca_total,prix,prix_moyen,prix_min,prix_max
0,0,0,0_0,1242,4657.50,3.75,3.75,3.75,3.75
1,0,1,0_1,487,5352.13,10.99,10.99,10.99,10.99
2,0,2,0_2,19,311.60,16.40,16.40,16.40,16.40
3,0,3,0_3,14,405.86,28.99,28.99,28.99,28.99
4,0,4,0_4,9,227.97,25.33,25.33,25.33,25.33


In [104]:
# Ordina i prodotti per CA crescente
lorenz_df = analyse_prix.sort_values('ca_total').reset_index(drop=True)

# Calcola percentuali cumulate
lorenz_df['ca_cumule'] = lorenz_df['ca_total'].cumsum()
lorenz_df['ca_cumule_pct'] = lorenz_df['ca_cumule'] / lorenz_df['ca_total'].sum() * 100
lorenz_df['produits_cumule_pct'] = (np.arange(1, len(lorenz_df)+1) / len(lorenz_df)) * 100
lorenz_df = lorenz_df.drop(['prix_min', 'prix_max'], axis=1)

lorenz_df.tail(5)

,category_id,product_num,produit_id,nb_ventes,ca_total,prix,prix_moyen,ca_cumule,ca_cumule_pct,produits_cumule_pct
3260,2,209,2_209,814,56971.86,69.99,69.99,11737290.11,97.585790,99.877489
3261,2,102,2_102,1027,60736.78,59.14,59.14,11798026.89,98.090766,99.908116
3262,2,112,2_112,968,65407.76,67.57,67.57,11863434.65,98.634577,99.938744
3263,2,135,2_135,1005,69334.95,68.99,68.99,11932769.60,99.211040,99.969372
3264,2,159,2_159,650,94893.50,145.99,145.99,12027663.10,100.000000,100.000000


In [105]:
#COURBE DE LORENZ par produit
fig = go.Figure()
fig.add_trace(go.Scatter(x=lorenz_df['produits_cumule_pct'], y=lorenz_df['ca_cumule_pct'],
                         mode='lines', name='Courbe de Lorenz'))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines', 
                         name='Égalité parfaite', line=dict(dash='dash')))
fig.update_layout(title='Courbe de Lorenz — Concentration du CA par produit',
                  xaxis_title='% cumulé des produits',
                  yaxis_title='% cumulé du CA')
fig.show()


In [106]:
#indice GINI
y = lorenz_df['ca_total'].values
n = len(y)
i = np.arange(1, n + 1)

G = (2 * np.sum(i * y)) / (n * np.sum(y)) - (n + 1) / n
print(f"Indice di Gini: {G:.4f}")

Indice di Gini: 0.7440


In [107]:
# Pareto 80/20
# Calcola la soglia del 80% del CA totale

# 3 — Trova il punto 80/20
seuil_80 = lorenz_df[lorenz_df['ca_cumule_pct'] <= 80]
print(f"{len(seuil_80)} produits sur {len(lorenz_df)} génèrent 80% du CA")
print(f"Soit {round(len(seuil_80)/len(lorenz_df)*100, 1)}% des produits")

print("Contrairement au principe de Pareto classique, notre catalogue est très équilibré : 98% des produits contribuent à 80% du CA. Cela signifie qu'aucun produit n'est dispensable — chaque référence joue un rôle dans le chiffre d'affaires. La stratégie doit donc viser à maintenir la diversité du catalogue plutôt que de se concentrer sur quelques best-sellers.")


3210 produits sur 3265 génèrent 80% du CA
Soit 98.3% des produits
Contrairement au principe de Pareto classique, notre catalogue est très équilibré : 98% des produits contribuent à 80% du CA. Cela signifie qu'aucun produit n'est dispensable — chaque référence joue un rôle dans le chiffre d'affaires. La stratégie doit donc viser à maintenir la diversité du catalogue plutôt que de se concentrer sur quelques best-sellers.


In [108]:
#fig pareto

fig_pareto = px.bar(lorenz_df.tail(20), x='produit_id', y='ca_total',
             title='Pareto — Top 20 produits ')
fig_pareto.show()

<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">C - Clients et catégories des livres</h4>
</div>

In [109]:
#lien entre le genre d’un client et les catégories des livres achetés,
# prendere transaction b2c e aggiungere costumers per avere le informazioni sul genere dei clienti
transactions_b2c = transactions_b2c.merge(costumers[['client_id', 'sex']], on='client_id', how='left')
transactions_b2c.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num,tranche_prix,sex
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259,10-20€,f
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390,10-20€,m
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352,0-10€,m
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110,0-10€,m
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638,0-10€,m


In [110]:
transactions_b2c.head(5)

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num,tranche_prix,sex
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259,10-20€,f
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390,10-20€,m
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352,0-10€,m
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110,0-10€,m
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638,0-10€,m


In [111]:
#cosa acquistano le persone di sex M partendo da transactions_b2c
transactions_b2c_m = transactions_b2c[transactions_b2c['sex'] == 'm']
#sex F da trasaction_b2c
transactions_b2c_f = transactions_b2c[transactions_b2c['sex'] == 'f']

In [112]:
# Distribution des ventes par catégorie pour les hommes
cat_distribution_m = transactions_b2c_m['category_id'].value_counts().reset_index()
cat_distribution_m.columns = ['category_id', 'count']
# Distribution des ventes par catégorie pour les femmes
cat_distribution_f = transactions_b2c_f['category_id'].value_counts().reset_index()
cat_distribution_f.columns = ['category_id', 'count']

In [113]:
fig_cat_dist_m = px.pie(cat_distribution_m, 
             values='count', 
             names='category_id',
             title='Distribution des ventes par catégorie M',
             hole=0.4)
fig_cat_dist_m.show()

fig_cat_dist_f = px.pie(cat_distribution_f,
                        values='count',
                        names='category_id',
                        title= 'Distribution des vente par catégorie F',
                        hole= 0.4
                        )
fig_cat_dist_f.show()

<div style="background-color: RGB(0,128,128);" >
<h5 style="margin: auto; padding: 20px; color:#fff; ">C.1 T-test genre et cat</h5>
</div>

In [114]:
#t-test genre x n°comandes

uomini = transactions_b2c[transactions_b2c['sex'] == 'm']['client_id'].value_counts()
donne = transactions_b2c[transactions_b2c['sex'] == 'f']['client_id'].value_counts()

t_stat, p_value = stats.ttest_ind(uomini, donne)
print(f't-statistic: {round(t_stat, 4)}')
print(f'p-value: {round(p_value, 4)}')

if p_value < 0.05:
    print('Differenza significativa ✅')
else:
    print('Differenza NON significativa ❌')

# Testo dinamico con variabili reali
print(f"\nLe test t de Student (t={round(t_stat, 2)}, p={round(p_value, 2)}) "
      f"ne révèle aucune différence significative entre le nombre d'achats "
      f"des hommes et des femmes (p > 0.05). "
      f"Le genre n'est donc pas un facteur discriminant du comportement d'achat.")

t-statistic: 0.092
p-value: 0.9267
Differenza NON significativa ❌

Le test t de Student (t=0.09, p=0.93) ne révèle aucune différence significative entre le nombre d'achats des hommes et des femmes (p > 0.05). Le genre n'est donc pas un facteur discriminant du comportement d'achat.


In [190]:
transactions_b2c.head()

,id_prod,date,session_id,client_id,age,tranche_age,price,year,month_num,month,day,week,year_week,category_id,product_num,tranche_prix,sex
0,0_1259,2021-03-01,1,329,56,50-64 Pré-retraités,11.99,2021,3,March,1,9,2021-S09,0,1259,10-20€,f
1,0_1390,2021-03-01,2,664,63,50-64 Pré-retraités,19.37,2021,3,March,1,9,2021-S09,0,1390,10-20€,m
2,0_1352,2021-03-01,3,580,35,35-49 Adultes,4.50,2021,3,March,1,9,2021-S09,0,1352,0-10€,m
3,0_1110,2021-03-01,3,580,35,35-49 Adultes,4.71,2021,3,March,1,9,2021-S09,0,1110,0-10€,m
4,0_1638,2021-03-01,3,580,35,35-49 Adultes,5.46,2021,3,March,1,9,2021-S09,0,1638,0-10€,m


In [191]:
# 1. Creazione della tabella di contingenza (Tabella incrociata Genere x Categoria)
contingency_table = pd.crosstab(
    transactions_b2c["sex"], transactions_b2c["category_id"]
)

print("--- Tabella di Contingenza ---")
print(contingency_table)
print("\n" + "=" * 40 + "\n")

# 2. Calcolo del Test del Chi-Quadrato
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

# 3. Stampa dei risultati
print(f"Chi2 statistic: {round(chi2, 4)}")
print(f"p-value: {round(p_value, 4)}")
print(f"Gradi di libertà (dof): {dof}")

if p_value < 0.05:
    print("Dipendenza significativa tra Genere e Categoria ✅")
else:
    print("Nessuna dipendenza significativa (Indipendenza) ❌")

# 4. Testo di sintesi dinamico
print(
    f"\nLe test du Chi-deux (χ²={round(chi2, 2)}, p={round(p_value, 4)}) "
    f"{'révèle' if p_value < 0.05 else 'ne révèle'} une relation "
    f"{'statistiquement significative' if p_value < 0.05 else 'non significative'} "
    f"entre le genre du client et la catégorie d'article achetée."
)

--- Tabella di Contingenza ---
category_id       0       1      2
sex                               
f            200793  115721  16980
m            186488  104884  15868


Chi2 statistic: 22.6686
p-value: 0.0
Gradi di libertà (dof): 2
Dipendenza significativa tra Genere e Categoria ✅

Le test du Chi-deux (χ²=22.67, p=0.0) révèle une relation statistiquement significative entre le genre du client et la catégorie d'article achetée.


In [ ]:
# V < 0.10: Associazione trascurabile / quasi nulla 
# V < 0.30: Associazione debole0.30
# V < 0.50: Associazione media 
# V>= 0.50: Associazione forte
# Calcolo della V di Cramér
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramer_v = np.sqrt(chi2 / (n * min_dim))

print(f"V di Cramér: {round(cramer_v, 4)}")

V di Cramér: 0.0059


<div style="background-color: RGB(0,128,128);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">Top achat B2B</h4>
</div>

In [115]:
# Acquisti b2b top 
# Filtra solo transazioni B2B
transactions_b2b = transactions[transactions['client_id'].isin(liste_b2b)]

# Prodotti più acquistati dai B2B
top_produits_b2b = (transactions_b2b.groupby(['id_prod', 'category_id']).agg(
    nb_ventes=('price', 'count'),
    ca_total=('price', 'sum')
).reset_index()
 .sort_values('nb_ventes', ascending=False))

top_produits_b2b.head(10)

,id_prod,category_id,nb_ventes,ca_total
1858,1_403,1,166,2986.34
1921,1_498,1,160,3739.20
1851,1_395,1,153,4435.47
1867,1_417,1,146,3064.54
1857,1_400,1,141,2340.60
1844,1_383,1,141,4087.59
1870,1_425,1,137,2327.63
1859,1_406,1,136,3374.16
1850,1_392,1,136,2462.96
1834,1_369,1,135,3238.65


In [116]:
print(top_produits_b2b)

     id_prod  category_id  nb_ventes  ca_total
1858   1_403            1        166   2986.34
1921   1_498            1        160   3739.20
1851   1_395            1        153   4435.47
1867   1_417            1        146   3064.54
1857   1_400            1        141   2340.60
...      ...          ...        ...       ...
534    0_160            0          1      9.12
1008  0_2143            0          1     24.99
2205   2_133            2          1    109.99
1530   0_815            0          1     10.99
1571   0_897            0          1      9.99

[2355 rows x 4 columns]


In [117]:
#acquisti fatti piú spesso dai b2b
for cliente in liste_b2b:
    print(f'--- {cliente} ---')
    df_c = transactions_b2b[transactions_b2b['client_id'] == cliente]
    print(df_c.groupby(['id_prod', 'category_id'])['price']
              .count()
              .reset_index(name='nb_ventes')
              .sort_values('nb_ventes', ascending=False)
              .head(5))
    print()

--- 4958 ---
    id_prod  category_id  nb_ventes
436    2_39            2         94
434    2_37            2         93
305   2_110            2         89
303   2_109            2         85
397   2_209            2         84

--- 6714 ---
     id_prod  category_id  nb_ventes
1139   1_498            1         48
1092   1_403            1         42
1079   1_383            1         39
1091   1_400            1         38
1102   1_425            1         37

--- 1609 ---
    id_prod  category_id  nb_ventes
384  0_1438            0         95
311  0_1360            0         79
288  0_1334            0         78
372  0_1424            0         75
360  0_1411            0         73

--- 3454 ---
    id_prod  category_id  nb_ventes
853   1_403            1         61
847   1_395            1         52
832   1_369            1         47
859   1_414            1         46
843   1_388            1         44



In [118]:
fig = px.bar(top_produits_b2b.head(20), 
             x='id_prod', y='nb_ventes',
             color='category_id',
             title='Top 20 produits achetés par les B2B')
fig.show()

<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">D - Age et montant</h4>
</div>

In [119]:

#lien entre l'âge des clients et le montant total des achats,
costumers.head(5)

,client_id,sex,birth,age,tranche_age
0,4410,f,1967,56,50-64 Pré-retraités
1,7839,f,1975,48,35-49 Adultes
2,1699,f,1984,39,35-49 Adultes
3,5961,f,1962,61,50-64 Pré-retraités
4,5320,m,1943,80,65+ Retraités


In [120]:
#ajouter age clients b2b
costumers_b2b = costumers[costumers['client_id'].isin(clients_b2b['client_id'])]
costumers_b2b.head(5)
#aggiungere totale delle loro spese
total_spent_b2b = transactions_b2b.groupby('client_id')['price'].sum().reset_index(name='total_spent')
costumers_b2b = costumers_b2b.merge(total_spent_b2b, on='client_id', how='left')


In [121]:
#reordonner par total_spent
costumers_b2b.sort_values('total_spent', ascending=False).head(10)

,client_id,sex,birth,age,tranche_age,total_spent
2,1609,m,1980,43,35-49 Adultes,326039.89
0,4958,m,1999,24,18-24 Étudiants,290227.03
1,6714,f,1968,55,50-64 Pré-retraités,153918.60
3,3454,m,1969,54,50-64 Pré-retraités,114110.57


In [122]:
#graphique scatter plot pour voir rélation entre age et montant total des achats
#df clients b2b

fig_age_b2b = px.scatter(costumers_b2b, x='age', y='total_spent',
                        title='Relation entre l\'âge des clients B2B et le montant total des achats',
                        labels={'age': 'Âge', 'total_price': 'Montant total des achats'})

fig_age_b2b.show()

In [123]:
#costumers b2c totale delle spese
costumers_b2c = costumers[~costumers['client_id'].isin(clients_b2b['client_id'])]
total_spent_b2c = transactions_b2c.groupby('client_id')['price'].sum().reset_index(name='total_spent')
costumers_b2c = costumers_b2c.merge(total_spent_b2c, on='client_id', how='left')


In [124]:
# graphique costumers b2c spese x age
fig_age_b2c = px.scatter(costumers_b2c, x='age', y='total_spent',
                        title='Relation entre l\'âge des clients B2C et le montant total des achats',
                        labels={'age': 'Âge', 'Montant': 'Montant total des achats'})

fig_age_b2c.show()

In [125]:
df_panier.head()

,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
0,1,11.99,1,329,56,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
1,2,19.37,1,664,63,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
2,3,14.67,3,580,35,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
3,4,20.75,2,7912,34,25-34 Jeunes professionnels,2021-03-01,1,9,March,2021,3,2021-S09
4,5,16.49,1,2033,67,65+ Retraités,2021-03-01,1,9,March,2021,3,2021-S09


In [126]:
#df_panier_age sans le b2b
df_panier_age_b2c = df_panier[~df_panier['client_id'].isin(liste_b2b)]

#graphique de dispersion entre l'âge des clients et le montant total des achats
fig_age_b2c_panier = px.scatter(df_panier_age_b2c, x='age', y='total_price',
                        title='Relation entre l\'âge des clients et le montant total des achats',
                        labels={'age': 'Âge', 'total_price': 'Montant total des achats'})


In [127]:
# Total spent per tranche d'age
spent_age = transactions_b2c.groupby('tranche_age')['price'].sum().reset_index()
spent_age.columns = ['tranche_age', 'total_spent']
spent_age['total_spent'] = spent_age['total_spent'].round(2)

# Grafico
fig = px.bar(spent_age, x='tranche_age', y='total_spent',
             title='Total dépensé par tranche d\'âge (B2C)',
             labels={'total_spent': 'Total dépensé (€)', 'tranche_age': 'Tranche d\'âge'},
             category_orders={'tranche_age': ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']})
fig.show()

In [128]:
#df_panier_age B2B
df_panier_age_b2b = df_panier[df_panier['client_id'].isin(liste_b2b)]
#graphique de dispersion entre l'âge des clients B2B et le montant total des achats
fig_age_b2b_panier = px.scatter(df_panier_age_b2b, x='age', y='total_price',
                        title='Relation entre l\'âge des clients B2B et le montant total des achats',
                        labels={'age': 'Âge', 'total_price': 'Montant total des achats'})
df_panier_age_b2b.head()

,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
6,7,13.85,2,1609,43,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
32,33,10.99,1,3454,54,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
45,46,16.98,2,1609,43,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
52,53,37.64,2,1609,43,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
62,63,24.81,1,1609,43,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09


In [129]:
fig_age_b2b_panier.show()

In [130]:
# CA par tranche d'âge et catégorie
spent_age_cat = (transactions_b2c.groupby(['tranche_age', 'category_id'])['price']
                 .sum()
                 .reset_index())
spent_age_cat.columns = ['tranche_age', 'category_id', 'total_spent']
spent_age_cat['total_spent'] = spent_age_cat['total_spent'].round(2)

# Grafico
fig = px.bar(spent_age_cat,
             x='tranche_age',
             y='total_spent',
             color='category_id',
             barmode='group',
             title='Catégorie de livres achetés par tranche d\'âge (B2C)',
             labels={'total_spent': 'Total dépensé (€)',
                     'tranche_age': 'Tranche d\'âge',
                     'category_id': 'Catégorie'},
             category_orders={'tranche_age': ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']})
fig.show()

In [131]:
fig = px.bar(spent_age_cat,
             x='tranche_age',
             y='total_spent',
             color='category_id',
             barmode='stack',
             text='category_id',
             title='Répartition des catégories par tranche d\'âge (B2C)',
             category_orders={'tranche_age': ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']})
fig.update_layout(barnorm='percent')  # ← normalizza a 100%
fig.show()

<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">E - Age et frequence d'achat</h4>
</div>

In [132]:
#lien entre l'âge des clients et la fréquence d’achat, B2C df_panier_age_b2c
df_panier_age_b2c_freq = df_panier_age_b2c.groupby('client_id').agg(
    age=('age', 'first'),
    freq_achat=('session_id', 'count')
).reset_index()


In [133]:
#lien entre l'âge des clients et la fréquence d’achat, B2B df_panier_age_b2b
df_panier_age_b2b_freq = df_panier_age_b2b.groupby('client_id').agg(
    age=('age', 'first'),
    freq_achat=('session_id', 'count')
).reset_index()


In [134]:
#grafico scatter plot pour voir rélation entre age et fréquence d’achat B2C
fig_age_b2c_freq = px.scatter(df_panier_age_b2c_freq, x='age', y='freq_achat',
                        title='Relation entre l\'âge des clients B2C et la fréquence d’achat',
                        labels={'age': 'Âge', 'freq_achat': 'Fréquence d’achat'})
#fig_age_b2c_freq.show()

In [135]:
#grafico scatter plot pour voir rélation entre age et fréquence d’achat B2B
fig_age_b2b_freq = px.scatter(df_panier_age_b2b_freq, x='age', y='freq_achat',
                        title='Relation entre l\'âge des clients B2B et la fréquence d’achat',
                        labels={'age': 'Âge', 'freq_achat': 'Fréquence d’achat'})
#fig_age_b2b_freq.show()


In [136]:
# Frequenza di acquisto per età — numero di sessioni per cliente
freq_age = df_panier_b2c.groupby('client_id').agg(
    nb_sessions=('session_id', 'count'),
    age=('age', 'first'),
    tranche_age=('tranche_age', 'first')
).reset_index()

# Media frequenza per tranche d'age
freq_par_tranche = (freq_age.groupby('tranche_age')['nb_sessions']
                    .mean()
                    .reset_index()
                    .rename(columns={'nb_sessions': 'freq_moyenne'})
                    .round(2))

print(freq_par_tranche)

                   tranche_age  freq_moyenne
0              18-24 Étudiants         19.10
1  25-34 Jeunes professionnels         27.40
2                35-49 Adultes         49.33
3          50-64 Pré-retraités         40.60
4                65+ Retraités         37.26


In [137]:
fig = px.bar(freq_par_tranche,
             x='tranche_age',
             y='freq_moyenne',
             title='Fréquence d\'achat moyenne par tranche d\'âge (B2C)',
             labels={'freq_moyenne': 'Nb sessions moyen', 
                     'tranche_age': 'Tranche d\'âge'},
                          category_orders={'tranche_age': ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']},
                        text='freq_moyenne')

fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.show()

In [138]:
from scipy.stats import shapiro, kruskal

tranches = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']
# Kruskal-Wallis frequenza vs tranche d'age
groupes_freq = [freq_age[freq_age['tranche_age'] == t]['nb_sessions'] 
                for t in tranches]

stat, p_value = kruskal(*groupes_freq)
print(f'Kruskal-Wallis stat: {round(stat, 4)}')
print(f'p-value: {round(p_value, 4)}')
print('Significativo ✅' if p_value < 0.05 else 'NON significativo ❌')

Kruskal-Wallis stat: 1090.0625
p-value: 0.0
Significativo ✅


In [139]:
# Media sessioni per età (non per tranche)
freq_age_scatter = df_panier_b2c.groupby('client_id').agg(
    nb_sessions=('session_id', 'count'),
    age=('age', 'first')
).reset_index()

# Media per età
freq_par_age = (freq_age_scatter.groupby('age')['nb_sessions']
                .mean()
                .reset_index()
                .rename(columns={'nb_sessions': 'freq_moyenne'})
                .round(2))

# Scatter
fig = px.scatter(freq_par_age,
                 x='age',
                 y='freq_moyenne',
                 title='Fréquence d\'achat moyenne par âge (B2C)',
                 labels={'freq_moyenne': 'Nb sessions moyen', 
                         'age': 'Âge'},
                 )  
fig.show()

<div style="background-color: RGB(0,150,20);" >
<h5 style="margin: auto; padding: 20px; color:#fff; "> Test Spearman - Age et frequence d'achat</h5>
</div>

In [140]:
# Età vs frequenza
from scipy.stats import spearmanr
corr, p = spearmanr(freq_age['age'], freq_age['nb_sessions'])
print(f'Spearman âge vs fréquence: r={round(corr, 4)}, p={round(p, 4)}')

Spearman âge vs fréquence: r=0.212, p=0.0


<div style="background-color: RGB(0,150,20);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">F - Age et montant du panier moyen</h4>
</div>

In [141]:
df_panier_b2b.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23039 entries, 6 to 345488
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   23039 non-null  int64         
 1   total_price  23039 non-null  float64       
 2   total_items  23039 non-null  int64         
 3   client_id    23039 non-null  int64         
 4   age          23039 non-null  int64         
 5   tranche_age  23039 non-null  object        
 6   date         23039 non-null  datetime64[ns]
 7   day          23039 non-null  int32         
 8   week         23039 non-null  UInt32        
 9   month        23039 non-null  object        
 10  year         23039 non-null  int32         
 11  month_num    23039 non-null  int32         
 12  year_week    23039 non-null  object        
dtypes: UInt32(1), datetime64[ns](1), float64(1), int32(3), int64(4), object(3)
memory usage: 2.1+ MB


<div style="background-color: RGB(0,128,128);" >
<h5 style="margin: auto; padding: 20px; color:#fff; ">Panier moyen B2B et B2C avec tranche d'age</h5>
</div>

In [142]:
#lien entre l'âge des clients et la taille du panier moyen (b2b)
panier_moyen_b2b = df_panier_b2b.groupby('client_id').agg(
    panier_moyen=('total_price', 'mean'),
    nb_sessions=('session_id', 'count'),    
    ca_total=('total_price', 'sum'),
    age=('age', 'first'),
    tranche_age=('tranche_age', 'first')
).reset_index().round(2).sort_values('panier_moyen', ascending=False)

panier_moyen_b2b.head(10)



,client_id,panier_moyen,nb_sessions,ca_total,age,tranche_age
2,4958,75.36,3851,290227.03,24,18-24 Étudiants
3,6714,58.75,2620,153918.60,55,50-64 Pré-retraités
0,1609,29.65,10997,326039.89,43,35-49 Adultes
1,3454,20.48,5571,114110.57,54,50-64 Pré-retraités


In [143]:
fig = px.bar(panier_moyen_b2b, 
             x='client_id', 
             y='panier_moyen',
             title='Panier moyen par client B2B',
             labels={'panier_moyen': 'Panier moyen (€)', 
                     'client_id': 'Client'},
             text='panier_moyen')

fig.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
fig.show()

In [144]:
#lien entre l'âge des clients et la taille du panier moyen (b2c)
panier_moyen_b2c = df_panier_b2c.groupby('client_id').agg(
    panier_moyen=('total_price', 'mean'),
    nb_sessions=('session_id', 'count'),
    ca_total=('total_price', 'sum'),
    age=('age', 'first'),
    tranche_age=('tranche_age', 'first')
).reset_index().round(2).sort_values('client_id', ascending=False)

panier_moyen_b2c.head(10)


,client_id,panier_moyen,nb_sessions,ca_total,age,tranche_age
8595,8621,29.90,117,3498.48,35,35-49 Adultes
8594,8620,47.86,10,478.64,22,18-24 Étudiants
8593,8619,33.29,81,2696.62,35,35-49 Adultes
8592,8618,80.43,39,3136.62,27,25-34 Jeunes professionnels
8591,8617,95.15,26,2474.02,27,25-34 Jeunes professionnels
8590,8616,30.77,96,2954.33,39,35-49 Adultes
8589,8615,30.37,54,1639.95,51,50-64 Pré-retraités
8588,8614,108.64,16,1738.16,20,18-24 Étudiants
8587,8613,25.49,41,1045.22,52,50-64 Pré-retraités
8586,8612,25.87,87,2250.79,58,50-64 Pré-retraités


In [145]:
tranches = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']

panier_moyen_par_tranche = (panier_moyen_b2c.groupby('tranche_age')['panier_moyen']
                             .mean()
                             .reset_index()
                             .round(2))

panier_moyen_par_tranche.columns = ['tranche_age', 'panier_moyen']

fig = px.bar(panier_moyen_par_tranche,
             x='tranche_age',
             y='panier_moyen',
             title='Panier moyen par tranche d\'âge (B2C)',
             labels={'panier_moyen': 'Panier moyen (€)', 
                     'tranche_age': 'Tranche d\'âge'},
             category_orders={'tranche_age': tranches},
             text='panier_moyen')

fig.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
fig.show()

<div style="background-color: RGB(0,128,128);" >
<h5 style="margin: auto; padding: 20px; color:#fff; ">Test Kruskal-Wallis - normalità e similarità negli acquisiti a secondo dell'età</h5>
</div>

In [146]:
df_panier_b2c.head(5)

,session_id,total_price,total_items,client_id,age,tranche_age,date,day,week,month,year,month_num,year_week
0,1,11.99,1,329,56,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
1,2,19.37,1,664,63,50-64 Pré-retraités,2021-03-01,1,9,March,2021,3,2021-S09
2,3,14.67,3,580,35,35-49 Adultes,2021-03-01,1,9,March,2021,3,2021-S09
3,4,20.75,2,7912,34,25-34 Jeunes professionnels,2021-03-01,1,9,March,2021,3,2021-S09
4,5,16.49,1,2033,67,65+ Retraités,2021-03-01,1,9,March,2021,3,2021-S09


In [147]:
conditions = [
    costumers['age'] < 25,
    costumers['age'] < 35,
    costumers['age'] < 50,
    costumers['age'] < 65,
]

labels = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités']

costumers['tranche_age'] = np.select(conditions, labels, default='65+ Retraités')

print(costumers['tranche_age'].value_counts())

tranche_age
35-49 Adultes                  2633
50-64 Pré-retraités            1991
25-34 Jeunes professionnels    1566
65+ Retraités                  1276
18-24 Étudiants                1155
Name: count, dtype: int64


In [148]:
from scipy.stats import shapiro, kruskal
from scipy.stats import mannwhitneyu
from itertools import combinations

tranches = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']

# 1 — Verifica normalità
print('--- Shapiro-Wilk ---')
for tranche in tranches:
    data = df_panier_b2c[df_panier_b2c['tranche_age'] == tranche]['total_price']
    print(f'{tranche} → n={len(data)}')
    stat, p = shapiro(data.sample(min(1000, len(data)), random_state=42))
    print(f'  p-value: {round(p, 4)} → {"normale ✅" if p > 0.05 else "NON normale ❌"}')

print()

# 2 — Kruskal-Wallis
print('--- Kruskal-Wallis ---')
groupes = [df_panier_b2c[df_panier_b2c['tranche_age'] == t]['total_price'] 
           for t in tranches]
stat, p_value = kruskal(*groupes)
print(f'stat: {round(stat, 4)}')
print(f'p-value: {round(p_value, 4)}')
print('Significativo ✅' if p_value < 0.05 else 'NON significativo ❌')

print()

# 3 — Mann-Whitney post-hoc
print('--- Mann-Whitney ---')
n_comparisons = len(list(combinations(tranches, 2)))
soglia = 0.05 / n_comparisons
print(f'Soglia Bonferroni: {round(soglia, 4)}\n')

for t1, t2 in combinations(tranches, 2):
    g1 = df_panier_b2c[df_panier_b2c['tranche_age'] == t1]['total_price']
    g2 = df_panier_b2c[df_panier_b2c['tranche_age'] == t2]['total_price']
    stat, p = mannwhitneyu(g1, g2)
    print(f'{t1} vs {t2} → p-value: {round(p, 4)} {"✅" if p < soglia else "❌"}')

print("Le test de Kruskal-Wallis révèle une différence hautement significative du panier moyen entre toutes les tranches d'âge (stat=40784, p<0.05). Les comparaisons par paires (Mann-Whitney avec correction de Bonferroni) confirment que chaque tranche d'âge a un comportement d'achat significativement différent des autres — l'âge est donc un facteur discriminant fort pour cibler les clients.")

--- Shapiro-Wilk ---
18-24 Étudiants → n=21889
  p-value: 0.0 → NON normale ❌
25-34 Jeunes professionnels → n=42769
  p-value: 0.0 → NON normale ❌
35-49 Adultes → n=129830
  p-value: 0.0 → NON normale ❌
50-64 Pré-retraités → n=80542
  p-value: 0.0 → NON normale ❌
65+ Retraités → n=47436
  p-value: 0.0 → NON normale ❌

--- Kruskal-Wallis ---
stat: 40784.5569
p-value: 0.0
Significativo ✅

--- Mann-Whitney ---
Soglia Bonferroni: 0.005

18-24 Étudiants vs 25-34 Jeunes professionnels → p-value: 0.0 ✅
18-24 Étudiants vs 35-49 Adultes → p-value: 0.0 ✅
18-24 Étudiants vs 50-64 Pré-retraités → p-value: 0.0 ✅
18-24 Étudiants vs 65+ Retraités → p-value: 0.0 ✅
25-34 Jeunes professionnels vs 35-49 Adultes → p-value: 0.0 ✅
25-34 Jeunes professionnels vs 50-64 Pré-retraités → p-value: 0.0 ✅
25-34 Jeunes professionnels vs 65+ Retraités → p-value: 0.0 ✅
35-49 Adultes vs 50-64 Pré-retraités → p-value: 0.0 ✅
35-49 Adultes vs 65+ Retraités → p-value: 0.0 ✅
50-64 Pré-retraités vs 65+ Retraités → p-value: 0

In [ ]:
from scipy.stats import shapiro, kruskal, mannwhitneyu
from itertools import combinations
import numpy as np

tranches = ['18-24 Étudiants', '25-34 Jeunes professionnels', '35-49 Adultes', '50-64 Pré-retraités', '65+ Retraités']

#  1. Verifica normalità (Shapiro) 
print('--- Shapiro-Wilk ---')
for tranche in tranches:
    data = df_panier_b2c[df_panier_b2c['tranche_age'] == tranche]['total_price']
    n = len(data)
    print(f'{tranche} → n={n}')
    # campione limitato a 1000 per evitare rallentamenti (e perché su grandi n è sempre significativo)
    sample = data.sample(min(1000, n), random_state=42)
    stat, p = shapiro(sample)
    # Formattazione p-value: se è zero, stampa "0.0" oppure in notazione scientifica
    p_str = f'{p:.4f}' if p > 0.0001 else f'{p:.2e}'
    print(f'  p-value: {p_str} → {"normale ✅" if p > 0.05 else "NON normale ❌"}')
print()

#  2. Kruskal-Wallis (con effect size) 
print('--- Kruskal-Wallis ---')
groupes = [df_panier_b2c[df_panier_b2c['tranche_age'] == t]['total_price'] 
           for t in tranches]
stat, p_value = kruskal(*groupes)
n_total = sum(len(g) for g in groupes)   # numero totale di osservazioni
k = len(tranches)                        # numero di gruppi

# Eta-quadrato (effect size) per Kruskal-Wallis
eta2 = (stat - k + 1) / (n_total - k)

print(f'Statistica H: {stat:.4f}')
print(f'p-value: {p_value:.4e}')
print(f'Significatif ✅' if p_value < 0.05 else 'NON significatif ❌')
print(f'Effect size (η²): {eta2:.4f}  → interpr.: <0.06 petit, 0.06‑0.14 moyen, >0.14 grand')
print()

# 3. Mann-Whitney post-hoc (con correzione di Bonferroni) 
print('--- Mann-Whitney ---')
n_comparisons = len(list(combinations(tranches, 2)))
soglia = 0.05 / n_comparisons
print(f'Soglia Bonferroni: {soglia:.4f}\n')

for t1, t2 in combinations(tranches, 2):
    g1 = df_panier_b2c[df_panier_b2c['tranche_age'] == t1]['total_price']
    g2 = df_panier_b2c[df_panier_b2c['tranche_age'] == t2]['total_price']
    u_stat, p = mannwhitneyu(g1, g2)
    # Opzionale: effect size r (rango-biseriale)
    r = 1 - (2 * u_stat) / (len(g1) * len(g2))
    print(f'{t1} vs {t2} → p-value: {p:.4e} {"✅" if p < soglia else "❌"}  (r = {abs(r):.3f})')

print("\n" + "="*70)
print("CONCLUSIONE:")
print("Le test de Kruskal-Wallis révèle une différence hautement significative du panier moyen")
print(f"entre toutes les tranches d'âge (H={stat:.1f}, p<0.001, η²={eta2:.3f}).")
print("Les comparaisons par paires (Mann-Whitney avec correction de Bonferroni) confirment")
print("que chaque tranche d'âge a un comportement d'achat significativement différent des autres.")
print(f"L'âge explique environ {eta2*100:.1f}% de la variance du panier moyen — c'est un facteur")
print("discriminant fort pour cibler les clients.")

--- Shapiro-Wilk ---
18-24 Étudiants → n=21889
  p-value: 3.94e-31 → NON normale ❌
25-34 Jeunes professionnels → n=42769
  p-value: 5.44e-37 → NON normale ❌
35-49 Adultes → n=129830
  p-value: 1.53e-31 → NON normale ❌
50-64 Pré-retraités → n=80542
  p-value: 1.18e-35 → NON normale ❌
65+ Retraités → n=47436
  p-value: 9.41e-37 → NON normale ❌

--- Kruskal-Wallis ---
Statistica H: 40784.5569
p-value: 0.0000e+00
Significatif ✅
Effect size (η²): 0.1265  → interpr.: <0.06 petit, 0.06‑0.14 moyen, >0.14 grand

--- Mann-Whitney ---
Soglia Bonferroni: 0.0050

18-24 Étudiants vs 25-34 Jeunes professionnels → p-value: 0.0000e+00 ✅  (r = 0.247)
18-24 Étudiants vs 35-49 Adultes → p-value: 0.0000e+00 ✅  (r = 0.507)
18-24 Étudiants vs 50-64 Pré-retraités → p-value: 0.0000e+00 ✅  (r = 0.630)
18-24 Étudiants vs 65+ Retraités → p-value: 0.0000e+00 ✅  (r = 0.653)
25-34 Jeunes professionnels vs 35-49 Adultes → p-value: 0.0000e+00 ✅  (r = 0.261)
25-34 Jeunes professionnels vs 50-64 Pré-retraités → p-value:

In [150]:
import plotly.express as px
import pandas as pd

# 1. Definisci l'ordine desiderato UNA VOLTA (dalla più giovane alla più anziana)
ordine_eta = [
    '18-24 Étudiants',
    '25-34 Jeunes professionnels',
    '35-49 Adultes',
    '50-64 Pré-retraités',
    '65+ Retraités'
]

# 2. Crea il violin plot forzando l'ordine con category_orders
fig = px.violin(
    df_panier_b2c,
    x='tranche_age',
    y='total_price',
    color='tranche_age',            # Colora ogni violino in modo diverso
    category_orders={'tranche_age': ordine_eta},  # <-- IMPOSTA L'ORDINE QUI
    box=True,                       # Mostra il boxplot all'interno
    points=False,                   # Nasconde i singoli punti (troppi dati)
    title='Distribution du panier moyen par tranche d\'âge',
    labels={'total_price': 'Panier moyen (€)', 'tranche_age': 'Tranche d\'âge'},
    template='plotly_white'
)

# 3. (Opzionale) Aggiungi la media globale come riferimento
media_globale = df_panier_b2c['total_price'].mean()
fig.add_hline(
    y=media_globale,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Moyenne globale: {media_globale:.2f} €',
    annotation_position='bottom right'
)

# 4. Layout
fig.update_layout(
    height=500,
    width=900,
    showlegend=False,               
    xaxis_title='Tranche d\'âge',
    yaxis_title='Panier moyen (€)',
    font=dict(size=12)
)

fig.show()

<div style="background-color: RGB(0,150,2);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">G - Age et catégorie de livres</h4>
</div>

In [151]:
#lien entre tranche âge des clients et la catégorie des livres achetés

# Total spent per tranche d'age e categoria
spent_age_cat = (transactions_b2c.groupby(['tranche_age', 'category_id'])['price']
                 .sum()
                 .reset_index())
spent_age_cat.columns = ['tranche_age', 'category_id', 'total_spent']
spent_age_cat['total_spent'] = spent_age_cat['total_spent'].round(2)

# Grafico
fig = px.bar(spent_age_cat, 
             x='tranche_age', 
             y='total_spent', 
             color='category_id',
             barmode='group',
             title='Achats par catégorie et tranche d\'âge (B2C)',
             labels={'total_spent': 'Total dépensé (€)', 
                     'tranche_age': 'Tranche d\'âge',
                     'category_id': 'Catégorie'},
             category_orders={'tranche_age': ['18-24', '25-34', '35-49', '50-64', '65+']})
fig.show()

<div style="background-color: RGB(0,150,2);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">H - Produits dormants ou Longue traîne</h4>
</div>


In [152]:
print('Distribution CA par produit:')
print(analyse_prix['ca_total'].describe())
print()
print('Distribution nb_ventes par produit:')
print(analyse_prix['nb_ventes'].describe())
print()
print('Percentili CA:')
print(analyse_prix['ca_total'].quantile([0.05, 0.10, 0.25]))
print()
print('Percentili nb_ventes:')
print(analyse_prix['nb_ventes'].quantile([0.05, 0.10, 0.25]))

Distribution CA par produit:
count     3265.000000
mean      3683.817182
std       7502.222605
min          0.990000
25%        233.870000
50%        810.980000
75%       3458.000000
max      94893.500000
Name: ca_total, dtype: float64

Distribution nb_ventes par produit:
count    3265.000000
mean      210.577029
std       337.807282
min         1.000000
25%        17.000000
50%        57.000000
75%       242.000000
max      2340.000000
Name: nb_ventes, dtype: float64

Percentili CA:
0.05     22.928
0.10     78.786
0.25    233.870
Name: ca_total, dtype: float64

Percentili nb_ventes:
0.05     5.0
0.10     7.0
0.25    17.0
Name: nb_ventes, dtype: float64


In [153]:
# Produits qui ne vendent presque rien
produits_dormants = analyse_prix[
    (analyse_prix['ca_total'] <= 5) | 
    (analyse_prix['nb_ventes'] <= 5)
].sort_values('ca_total', ascending=True)

print(f'Nombre de produits dormants: {len(produits_dormants)}')
print(f'Sur un total de {len(analyse_prix)} produits ({round(len(produits_dormants)/len(analyse_prix)*100, 1)}%)')
produits_dormants.head(10)

Nombre de produits dormants: 218
Sur un total de 3265 produits (6.7%)


,category_id,product_num,produit_id,nb_ventes,ca_total,prix,prix_moyen,prix_min,prix_max
1529,0,1539,0_1539,1,0.99,0.99,0.99,0.99,0.99
1275,0,1284,0_1284,1,1.38,1.38,1.38,1.38,1.38
1640,0,1653,0_1653,2,1.98,0.99,0.99,0.99,0.99
803,0,807,0_807,1,1.99,1.99,1.99,1.99,1.99
537,0,541,0_541,1,1.99,1.99,1.99,1.99,1.99
1591,0,1601,0_1601,1,1.99,1.99,1.99,1.99,1.99
1715,0,1728,0_1728,1,2.27,2.27,2.27,2.27,2.27
1488,0,1498,0_1498,1,2.48,2.48,2.48,2.48,2.48
894,0,898,0_898,2,2.54,1.27,1.27,1.27,1.27
1825,0,1840,0_1840,2,2.56,1.28,1.28,1.28,1.28


In [154]:
# CA perdu si on les retire
ca_dormants = produits_dormants['ca_total'].sum()
ca_total_produits = analyse_prix['ca_total'].sum()
print(f'CA généré par les produits dormants: {round(ca_dormants, 2)}€')
print(f'Soit {round(ca_dormants/ca_total_produits*100, 2)}% du CA total')

print("6.7% du catalogue (218 références) génère moins de 5 ventes en 2 ans. Ces produits représentent 0.16% du CA — leur maintien en ligne a un coût (gestion, hébergement) sans retour significatif. Une rationalisation ciblée de ces références permettrait de concentrer les efforts marketing sur les produits performants")

CA généré par les produits dormants: 19844.89€
Soit 0.16% du CA total
6.7% du catalogue (218 références) génère moins de 5 ventes en 2 ans. Ces produits représentent 0.16% du CA — leur maintien en ligne a un coût (gestion, hébergement) sans retour significatif. Une rationalisation ciblée de ces références permettrait de concentrer les efforts marketing sur les produits performants


In [155]:
# Répartition des produits dormants par catégorie
dormants_cat = produits_dormants.groupby('category_id').agg(
    nb_produits=('nb_ventes', 'count'),
    ca_total=('ca_total', 'sum')
).reset_index()

print(dormants_cat)

# Graphique à pie
fig = px.pie(dormants_cat,
             values='nb_produits',
             names='category_id',
             title='Répartition des produits dormants par catégorie',
             labels={'nb_produits': 'Nb produits', 'category_id': 'Catégorie'},
             hole=0.4, 
             color_discrete_sequence=['blue', 'green', 'red']
             )
fig.show()

   category_id  nb_produits  ca_total
0            0          183  10957.19
1            1           16   1827.25
2            2           19   7060.45


In [156]:
# % produits dormants vs catalogue total per categoria
total_cat = analyse_prix.groupby('category_id').size().reset_index(name='total_produits')
dormants_cat = dormants_cat.merge(total_cat, on='category_id')
dormants_cat['pct_dormants'] = (dormants_cat['nb_produits'] / dormants_cat['total_produits'] * 100).round(2)
print(dormants_cat)

   category_id  nb_produits  ca_total  total_produits  pct_dormants
0            0          183  10957.19            2292          7.98
1            1           16   1827.25             737          2.17
2            2           19   7060.45             236          8.05


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Graphiques</h2>
</div>

<div style="background-color: RGB(220,0,300);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.1 - Graphiques</h4>
</div>

In [157]:
fig_11a = px.bar(ca_mensuel, x='month', y='total_ca', facet_col='year',
                category_orders={
                    'month': ['January','February','March','April','May','June',
                              'July','August','September','October','November','December'],
                    'year': sorted(ca_mensuel['year'].unique())  # anni in ordine
                },
                title='CA par mois et par année')
fig_11a.show()

In [158]:
#ca_week
fig_week = px.bar(ca_week, x='year_week', y='total_ca',
             title='CA par semaine et par année',
             labels={'total_ca': 'CA (€)', 'year_week': 'Semaine'})
fig_week.show()

In [159]:
# ca_day
fig_13 = px.line(ca_day, x = 'date', y = 'total_ca', title = 'CA par jour' )
fig_13.show()

In [160]:
ca_day['moyenne_mobile_7j'] = ca_day['total_ca'].rolling(window=7).mean()

fig_moy = px.line(ca_day, x='date', y= 'moyenne_mobile_7j',
              title='CA journalier + moyenne mobile 7 jours')
fig_moy.show()

In [161]:
ca_day_b2b['moyenne_mobile_7j'] = ca_day_b2b['total_price'].rolling(window=7).mean()
ca_day_b2c['moyenne_mobile_7j'] = ca_day_b2c['total_price'].rolling(window=7).mean()

fig_moy_b2c = px.line(ca_day_b2c, x='date', y='moyenne_mobile_7j',
              title='CA journalier b2c + moyenne mobile 7 jours')
fig_moy_b2c.show()
fig_moy_b2b = px.line(ca_day_b2b, x='date', y='moyenne_mobile_7j',
              title='CA journalier b2b + moyenne mobile 7 jours')
fig_moy_b2b.show()

In [162]:
fig_moy_b2b = px.line(ca_day_b2b, x='date', y='moyenne_mobile_7j',
              title='CA journalier b2b + moyenne mobile 7 jours')
fig_moy_b2b.show()

In [163]:
# Periodo del calo — ottobre 2021
periodo_calo = transactions[
    (transactions['date'] >= '2021-10-05') & 
    (transactions['date'] <= '2021-11-05')
]

# Stesso periodo un anno dopo — ottobre 2022
periodo_normale_2022 = transactions[
    (transactions['date'] >= '2022-10-05') & 
    (transactions['date'] <= '2022-11-05')
]

# Confronto
print('--- Periodo calo (oct 2021) ---')
print('Transactions/jour (moy.):', round(periodo_calo.groupby('date')['session_id'].count().mean(), 1))
print('CA/jour (moy.):', round(periodo_calo.groupby('date')['price'].sum().mean(), 2))
print('Prix moyen:', round(periodo_calo['price'].mean(), 2))

print()
print('--- Même période 2022 ---')
print('Transactions/jour (moy.):', round(periodo_normale_2022.groupby('date')['session_id'].count().mean(), 1))
print('CA/jour (moy.):', round(periodo_normale_2022.groupby('date')['price'].sum().mean(), 2))
print('Prix moyen:', round(periodo_normale_2022['price'].mean(), 2))

print()
# Variazione %
ca_2021 = periodo_calo.groupby('date')['price'].sum().mean()
ca_2022 = periodo_normale_2022.groupby('date')['price'].sum().mean()
print(f'Variation CA: {round((ca_2021 - ca_2022)/ca_2022*100, 1)}%')

--- Periodo calo (oct 2021) ---
Transactions/jour (moy.): 931.8
CA/jour (moy.): 15547.83
Prix moyen: 16.69

--- Même période 2022 ---
Transactions/jour (moy.): 933.2
CA/jour (moy.): 16524.12
Prix moyen: 17.71

Variation CA: -5.9%


In [164]:
#indagine calo ottobre 2021
# Isola il periodo del calo
periodo_calo = transactions[(transactions['date'] >= '2021-10-05') & 
                             (transactions['date'] <= '2021-11-05')]

# Quante transazioni al giorno?
print(periodo_calo.groupby('date')['session_id'].count())

# CA medio giornaliero nel periodo normale vs calo
normale = transactions[(transactions['date'] >= '2021-09-01') & 
                        (transactions['date'] <= '2021-10-17')]
ca_moyen_normal = round(normale.groupby('date')['price'].sum().mean(), 2)
ca_moyen_calo = round(periodo_calo.groupby('date')['price'].sum().mean(), 2)
print(ca_moyen_normal)
print(ca_moyen_calo)

# Il calo è nel prezzo medio o nel numero di transazioni?
print('Prezzo medio normale:', round(normale['price'].mean(), 2))
print('Prezzo medio calo:', round(periodo_calo['price'].mean(), 2))

print(f'Variation CA: {round((ca_moyen_calo - ca_moyen_normal)/ca_moyen_normal*100, 1)}%')

date
2021-10-05    1085
2021-10-06    1236
2021-10-07    1056
2021-10-08    1123
2021-10-09    1055
2021-10-10     968
2021-10-11    1037
2021-10-12     968
2021-10-13     930
2021-10-14     963
2021-10-15     964
2021-10-16     969
2021-10-17     898
2021-10-18     828
2021-10-19     827
2021-10-20     810
2021-10-21     801
2021-10-22     744
2021-10-23     787
2021-10-24     800
2021-10-25     721
2021-10-26     762
2021-10-27     713
2021-10-28     971
2021-10-29    1059
2021-10-30    1017
2021-10-31    1033
2021-11-01     935
2021-11-02     976
2021-11-03     907
2021-11-04     914
2021-11-05     961
Name: session_id, dtype: int64
17159.14
15547.83
Prezzo medio normale: 15.67
Prezzo medio calo: 16.69
Variation CA: -9.4%


In [165]:
# Il prezzo medio era più basso durante il picco del 6 ottobre?
print(transactions[transactions['date'] == '2021-10-06']['price'].mean().round(2))
print(transactions[transactions['date'] == '2021-10-27']['price'].mean().round(2))

16.27
17.54


In [166]:
# Chi ha smesso di comprare? B2B o B2C?
print(periodo_calo.groupby(['date', 'client_id']).size().groupby('date').count())

# Quali categorie hanno calato di più?
print(periodo_calo.groupby('category_id')['price'].sum())
normale_cat = normale.groupby('category_id')['price'].sum()
calo_cat = periodo_calo.groupby('category_id')['price'].sum()
print('\nDifferenza per categoria:')
print(((calo_cat - normale_cat) / normale_cat * 100).round(2))

date
2021-10-05    504
2021-10-06    529
2021-10-07    465
2021-10-08    486
2021-10-09    460
2021-10-10    449
2021-10-11    480
2021-10-12    443
2021-10-13    432
2021-10-14    432
2021-10-15    455
2021-10-16    457
2021-10-17    422
2021-10-18    415
2021-10-19    403
2021-10-20    400
2021-10-21    403
2021-10-22    369
2021-10-23    396
2021-10-24    401
2021-10-25    379
2021-10-26    378
2021-10-27    378
2021-10-28    407
2021-10-29    443
2021-10-30    415
2021-10-31    434
2021-11-01    441
2021-11-02    452
2021-11-03    445
2021-11-04    420
2021-11-05    425
dtype: int64
category_id
0    199733.81
1    200652.99
2     97143.74
Name: price, dtype: float64

Differenza per categoria:
category_id
0   -44.74
1   -39.11
2   -15.91
Name: price, dtype: float64


In [167]:
# Prezzo medio per categoria nel periodo normale vs calo
normale_cat = transactions[(transactions['date'].between('2021-09-01', '2021-10-04'))]
calo_cat = transactions[(transactions['date'].between('2021-10-18', '2021-10-27'))]

print('Prezzo medio per categoria — periodo normale:')
print(normale_cat.groupby('category_id')['price'].mean().round(2))

print()
print('Prezzo medio per categoria — periodo calo:')
print(calo_cat.groupby('category_id')['price'].mean().round(2))

Prezzo medio per categoria — periodo normale:
category_id
0    10.65
1    20.54
2    77.35
Name: price, dtype: float64

Prezzo medio per categoria — periodo calo:
category_id
0    10.55
1    20.66
2    75.11
Name: price, dtype: float64


In [168]:
# Numero vendite per categoria — periodo normale vs calo
print('Nb vendite per categoria — periodo normale:')
print(normale_cat.groupby('category_id')['price'].count())

print()
print('Nb vendite per categoria — periodo calo:')
print(calo_cat.groupby('category_id')['price'].count())

print()
# Differenza in %
nb_normale = normale_cat.groupby('category_id')['price'].count()
nb_calo = calo_cat.groupby('category_id')['price'].count()
print('Variazione % vendite:')
print(((nb_calo - nb_normale) / nb_normale * 100).round(2))

Nb vendite per categoria — periodo normale:
category_id
0    25715
1    11457
2     1039
Name: price, dtype: int64

Nb vendite per categoria — periodo calo:
category_id
0    5714
1    1677
2     402
Name: price, dtype: int64

Variazione % vendite:
category_id
0   -77.78
1   -85.36
2   -61.31
Name: price, dtype: float64


In [169]:
giorni_normale = (pd.Timestamp('2021-10-04') - pd.Timestamp('2021-09-01')).days
giorni_calo = (pd.Timestamp('2021-10-27') - pd.Timestamp('2021-10-18')).days

print('Vendite medie/giorno — normale:')
print((nb_normale / giorni_normale).round(1))

print()
print('Vendite medie/giorno — calo:')
print((nb_calo / giorni_calo).round(1))

Vendite medie/giorno — normale:
category_id
0    779.2
1    347.2
2     31.5
Name: price, dtype: float64

Vendite medie/giorno — calo:
category_id
0    634.9
1    186.3
2     44.7
Name: price, dtype: float64


<div style="background-color: RGB(220,0,300);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.2 CA et CAT</h4>
</div>

In [170]:
#CA par mois et catégorie - AVERE GRAFICO PER ANNO
order_mois = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']


fig_cat = px.bar(ca_mois_cat, x='month', y='ca', color='category_id', 
                 facet_col='category_id',
                 facet_row='year',
                 title="CA par mois et catégorie par année",
                 labels={'ca': 'CA (€)', 'month': 'Mois', 'category_id': 'Catégorie'},
                 category_orders={'month': order_mois})
fig_cat.show()

In [185]:
#distribution du CA par catégorie à l'année
fig_cat_year = px.pie(top_categories, 
                      values='ca', 
                      names='category_id',
                      title="CA par catégorie",
                      labels={'ca': 'CA (€)', 'category_id': 'Catégorie'},
                      hole=0.4,
                      color='category_id',  # Specifica la colonna su cui applicare i colori
                      color_discrete_map={
                          0: 'blue',   # Oppure un codice esadecimale come '#0000FF'
                          1: 'red',    # Oppure '#FF0000'
                          2: 'green'   # Oppure '#008000'
                      },
                      category_orders={'category_id': [0, 1, 2]})
fig_cat_year.show()

In [172]:
top_categories.head(10)

,category_id,nb_ventes,ca
0,0,415459,4419730.97
1,1,235592,4827657.11
2,2,36483,2780275.02


In [173]:
top_categories_b2b.head()

,category_id,nb_ventes,total_ca
0,0,28178,300530.28
1,1,14987,307555.25
2,2,3635,276210.56


In [174]:
#distribution du CA par catégorie à l'année
fig_cat_year = px.pie(top_categories_b2b, values='total_ca', names='category_id',
                      title="CA par catégorie B2B",
                      labels={'ca': 'CA (€)', 'category_id': 'Catégorie'},
                      hole=0.4)
fig_cat_year.show()

In [175]:
# variation CA par CAT entre B2B e totale
# CA totale per categoria
ca_cat_total = transactions.groupby('category_id')['price'].sum().reset_index(name='ca_total')

# CA B2B per categoria
ca_cat_b2b = transactions_b2b.groupby('category_id')['price'].sum().reset_index(name='ca_b2b')

# CA B2C per categoria
ca_cat_b2c = transactions_b2c.groupby('category_id')['price'].sum().reset_index(name='ca_b2c')

# Merge tutto insieme
ca_cat = ca_cat_total.merge(ca_cat_b2b, on='category_id', how='left')
ca_cat = ca_cat.merge(ca_cat_b2c, on='category_id', how='left')

# Calcola percentuali
ca_cat['pct_b2b'] = (ca_cat['ca_b2b'] / ca_cat['ca_total'] * 100).round(2)
ca_cat['pct_b2c'] = (ca_cat['ca_b2c'] / ca_cat['ca_total'] * 100).round(2)

ca_cat = ca_cat.round(2)
print(ca_cat)

   category_id    ca_total     ca_b2b      ca_b2c  pct_b2b  pct_b2c
0            0  4419730.97  300530.28  4119200.69     6.80    93.20
1            1  4827657.11  307555.25  4520101.86     6.37    93.63
2            2  2780275.02  276210.56  2504064.46     9.93    90.07


<div style="background-color: RGB(220,0,300);" >
<h4 style="margin: auto; padding: 20px; color:#fff; ">1.2 CA et prod</h4>
</div>

In [176]:
ca_mois_prod.head()

,year,month,id_prod,nb_ventes,ca
19047,2021,November,1_412,131,2181.15
36394,2022,January,1_369,129,3094.71
36501,2022,January,1_498,125,2921.25
6911,2021,December,1_417,125,2623.75
36432,2022,January,1_417,125,2623.75


In [177]:
order_mois = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']


for anno in sorted(transactions['year'].unique()):
    # Top 10 prodotti per CA per quell'anno
    top10_anno = (transactions[transactions['year'] == anno]
                  .groupby('id_prod')['price']
                  .sum()                           # ← sum per CA
                  .reset_index(name='ca')
                  .sort_values('ca', ascending=False)
                  .head(10)['id_prod']
                  .tolist())
    
    # Filtra ca_mois_prod per anno e top 10 di quell'anno
    df_anno = ca_mois_prod[
        (ca_mois_prod['year'] == anno) & 
        (ca_mois_prod['id_prod'].isin(top10_anno))
    ]
    
    fig = px.bar(df_anno, x='month', y='ca', color='id_prod',
                 title=f'CA par mois — Top 10 produits par CA — {anno}',
                 labels={'ca': 'CA (€)', 'month': 'Mois', 'id_prod': 'Produit'},
                 category_orders={'month': order_mois})
    fig.show()

In [178]:
#best sellers par mois
best_sellers = transactions.groupby('id_prod').agg(
    nb_ventes=('price', 'count'),
    ca=('price', 'sum')
).reset_index().sort_values('nb_ventes', ascending=False)

top10_bs = best_sellers.head(10)['id_prod'].tolist()
fig_prod = px.bar(ca_mois_prod[ca_mois_prod['id_prod'].isin(top10_bs)],
                  x='month', y='nb_ventes', color='id_prod',
                  title="CA par mois — Top 10 produits")
fig_prod.show()




In [179]:
# Top prodotti per anno
top_produits_year = (transactions.groupby(['year', 'id_prod', 'category_id']).agg(
    nb_ventes=('price', 'count'),
    ca_total=('price', 'sum')
).reset_index().sort_values(['year', 'nb_ventes'], ascending=[True, False]))

# Top 10 per anno
for anno in sorted(top_produits_year['year'].unique()):
    top10 = top_produits_year[top_produits_year['year'] == anno].head(10)
    
    fig = px.bar(top10, 
                 x='id_prod', 
                 y='nb_ventes',
                 color='category_id',
                 title=f'Top 10 produits — {anno}',
                 labels={'nb_ventes': 'Nb ventes', 
                         'id_prod': 'Produit',
                         'category_id': 'Catégorie'})
    fig.show()

In [180]:
analyse_prix.head(5)

,category_id,product_num,produit_id,nb_ventes,ca_total,prix,prix_moyen,prix_min,prix_max
0,0,0,0_0,1242,4657.50,3.75,3.75,3.75,3.75
1,0,1,0_1,487,5352.13,10.99,10.99,10.99,10.99
2,0,2,0_2,19,311.60,16.40,16.40,16.40,16.40
3,0,3,0_3,14,405.86,28.99,28.99,28.99,28.99
4,0,4,0_4,9,227.97,25.33,25.33,25.33,25.33


In [181]:
analyse_prix_fig = px.line(analyse_prix, x='produit_id', y='nb_ventes', color='category_id',
                           title='Nombre de ventes par produit et catégorie',)
analyse_prix_fig.show()

In [182]:
#COURBE DE LORENZ
fig = go.Figure()
fig.add_trace(go.Scatter(x=lorenz_df['produits_cumule_pct'], y=lorenz_df['ca_cumule_pct'],
                         mode='lines', name='Courbe de Lorenz'))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines', 
                         name='Égalité parfaite', line=dict(dash='dash')))
fig.update_layout(title='Courbe de Lorenz — Concentration du CA par produit',
                  xaxis_title='% cumulé des produits',
                  yaxis_title='% cumulé du CA')
fig.show()
